# ReportSPPR v4.2

Сборка от 22.09.2026. Шаблон отчёта: `template_v4.html` — имя задаётся параметром `TEMPLATE_FILE` в ячейке 1.

Что нового в 4.2 (минор, шаблон не менялся): проверки сходимости №3 и №4 считают по факту и видят «Н/Д»; экранирование HTML в шаблоне (autoescape); в историю пишутся инциденты недели; «Проверка релиза» и «Внутреннее тестирование на Предпроде» — рабочие статусы; все «Плохо» без обрезки; при сбое ИИ — нейтральная строка вместо текста ошибки; нет таблицы Logs — предупреждение вместо падения; «в среднем за день» — по рабочим дням. Основание — аудит 01.10, `docs/my-FT.md` 1.19.

Что нового в 4.1 (минор, шаблон `template_v4.html` не менялся): кэш разметки по id (Т-63), оценка `Н/Д` вместо `ХОРОШО` при сбое ИИ (Т-64), возраст открытых от даты выгрузки (Т-65), ключ DeepSeek только при этом провайдере (Т-66). Требования — `docs/my-FT.md`.

Нумерация: мажор — версия шаблона, минор — правки только в ноутбуке.

Порядок ячеек менять нельзя: они делят общие переменные.

# БЛОК 0 и 1: ГЛОБАЛЬНЫЕ ПАРАМЕТРЫ, АВТОРИЗАЦИЯ, ФИЛЬТР

In [ ]:
# =====================================================================
# [ЯЧЕЙКА 1] БЛОК 0 и 1: ГЛОБАЛЬНЫЕ ПАРАМЕТРЫ, АВТОРИЗАЦИЯ, ФИЛЬТР И ЗАГРУЗКА
# =====================================================================
import os
import io
import glob
import re
import time
import json
import math
import asyncio
import pandas as pd
import numpy as np
from pydantic import BaseModel, Field
from jinja2 import Template, Environment
from google import genai
from google.colab import files, userdata
from openai import AsyncOpenAI, OpenAI

# --- ПАРАМЕТРЫ ПРОВАЙДЕРА И МОДЕЛЕЙ ---
AI_PROVIDER = "deepseek" #@param ["google", "deepseek"]

# Google Gemini модели
GEMINI_BATCH_MODEL = 'gemini-3.6-flash'
GEMINI_SUMMARY_MODEL = 'gemini-3.1-pro'

# DeepSeek модели
DEEPSEEK_BATCH_MODEL = 'deepseek-chat'
DEEPSEEK_SUMMARY_MODEL = 'deepseek-reasoner'

# Автоматический выбор активных моделей
BATCH_MODEL = GEMINI_BATCH_MODEL if AI_PROVIDER == "google" else DEEPSEEK_BATCH_MODEL
SUMMARY_MODEL = GEMINI_SUMMARY_MODEL if AI_PROVIDER == "google" else DEEPSEEK_SUMMARY_MODEL

# ЧТО ИЗМЕНЕНО: размер пачки уменьшен с 40 до 15, добавлены лимиты ответа и текста
# ПОЧЕМУ: после добавления issue_summary / recommendation / suggested_type ответ
#         модели перестал влезать в лимит вывода — JSON обрывался на всех пачках
BATCH_SIZE = 15
MAX_CONCURRENCY = 4
SLEEP_TIME = 2
MAX_OUTPUT_TOKENS = 8192            # потолок ответа модели на одну пачку
PROMPT_TEXT_LIMIT = 1500            # обрезка desc/sol во входном промпте

# --- РЕЖИМ РАБОТЫ С GOOGLE ДИСКОМ ---
# ЧТО ИЗМЕНЕНО: Диск монтируется лениво, только когда он реально нужен
# ПОЧЕМУ: окно согласия Google выдаётся раз в сессию и отключить его нельзя,
#         но при повторных прогонах и при готовом кэше Диск не нужен вообще
DRIVE_MODE = "auto" #@param ["auto", "always", "never"]
CACHE_DIR = '/content/sppr_cache'          # локальный кэш шаблона и истории на время сессии

# --- ШАБЛОН ОТЧЁТА ---
# ЧТО ИЗМЕНЕНО: имя файла шаблона вынесено в параметр
# ПОЧЕМУ: чтобы не переименовывать template_v3.html в template.html при каждой версии
TEMPLATE_FILE = 'template_v4.html' #@param {type:"string"}   # версия шаблона = версия ноутбука

# --- РЕЖИМ ВЫБОРА ФАЙЛА ---
USE_FILE_PICKER = True #@param {type:"boolean"}

# --- ПАРАМЕТРЫ ТЕСТИРОВАНИЯ ---
TEST_SAMPLE_SIZE = 0

# --- ПАРАМЕТРЫ БИЗНЕС-ЛОГИКИ ---
WORK_HOURS = 8
TOP_INITIATORS_N = 5
TOP_LONGEST_N = 10
TOP_PROBLEMS_N = 5

# ----------------------------------------------------------------------
# ЧТО ИЗМЕНЕНО: убраны USER_APPROVAL_DAYS / APPROVAL_HOURS_DEDUCT /
#               WORK_START_HOUR / APPLY_APPROVAL_DEDUCT
# ПОЧЕМУ: SLA считается как (дата решения - дата регистрации) / 3,
#         вычет согласования и смещение по началу смены больше не применяются
# ----------------------------------------------------------------------
SLA_DIVIDER = 3                     # календарные часы -> рабочие (24 / 8 = 3)
OPEN_AGE_DAYS = 30                  # граница "открыто более / менее месяца"
FAST_SOLVE_HOURS = 8                # "решено за рабочий день"
LONG_SOLVE_HOURS = 40               # "долгострой" — больше рабочей недели

# --- СТАТУСЫ ВЫГРУЗКИ (поле st) ---
# ЧТО ИЗМЕНЕНО: новый блок
# ПОЧЕМУ: выгрузка теперь содержит открытые и отклонённые обращения,
#         анализ качества считается только по завершённым
CLOSED_STATUSES = ['Отработано, требует подтверждения', 'Закрыто']
REJECTED_STATUS = 'Отклонено'
INCIDENT_TYPE = 'Инцидент'

# ----------------------------------------------------------------------
# ЧТО ИЗМЕНЕНО: рабочие статусы, цвета типов и возрастные корзины
# ПОЧЕМУ: слайд открытых обращений делит их на "нормальный ход работы"
#         и "требует чистки", а плитка "Всего" красится по типам
# ----------------------------------------------------------------------
# Статусы нормального хода работы — в таблицу "на чистку" НЕ попадают
WORKFLOW_STATUSES = [
    'Планирование разработки', 'Уточнение', 'Анализ', 'Ожидает анализа',
    'Внутреннее тестирование', 'Разработка', 'Релиз установлен',
    # ----------------------------------------------------------------------
    # ЧТО ИЗМЕНЕНО (v4.2): добавлены 'Проверка релиза' и 'Внутреннее тестирование на Предпроде'
    # ПОЧЕМУ: статусы есть в выгрузке; без них обращения в этих статусах старше месяца ложно попадали «на чистку»
    # ----------------------------------------------------------------------
    'Проверка релиза', 'Внутреннее тестирование на Предпроде'
]
# Статусы, для которых строится распределение по возрасту
AGE_WATCH_STATUSES = ['Уточнение', 'Разработка', 'Внутреннее тестирование']

# Цвета типов обращений (плитка "Всего" и легенды)
TYPE_COLORS = {
    'Инцидент': '#ea4335',
    'Изменение данных в системе': '#fbbc04',
    'Консультация': '#34a853',
    'Права доступа': '#1a56db',
    'Дубль': '#9aa5b8',
}
TYPE_COLOR_DEFAULT = '#7c3aed'

# Справочник типов для проверки классификации (ячейка 3)
KNOWN_TYPES = ['Инцидент', 'Консультация', 'Изменение данных в системе', 'Права доступа', 'Дубль']

# --- ПАРАМЕТРЫ ВТОРИЧНОЙ ГРУППИРОВКИ ТЕМ (ЯЧЕЙКА 4) ---
ENABLE_TOPIC_GROUPING = True #@param {type:"boolean"}
TOPIC_GROUPS_MIN = 8                # ориентир для модели: не меньше стольких групп
TOPIC_GROUPS_MAX = 15               # ориентир для модели: не больше стольких групп
TOPIC_GROUPING_RETRIES = 3          # число попыток вызова при сбое/непарсибельном ответе

# --- ПАРАМЕТРЫ ФАЙЛОВОЙ СИСТЕМЫ ---
if USE_FILE_PICKER:
    TARGET_DIR = '/content/result'
else:
    SOURCE_DIR_VARIANTS = [
        '/content/drive/MyDrive/REPORTS/SPPR/source',
        '/content/drive/MyDrive/reports/sppr/source'
    ]
    TARGET_DIR = '/content/drive/MyDrive/REPORTS/SPPR/result'

# Зафиксировано: История сохраняется на Google Диск постоянной папки
HISTORY_DIR = '/content/drive/MyDrive/REPORTS/SPPR/result/history'

# --- ИНИЦИАЛИЗАЦИЯ ИИ КЛИЕНТОВ И ДИСКА ---
# 1. Google Gemini
GOOGLE_API_KEY = userdata.get('GEMINI_API_KEY')
client_gemini = genai.Client(api_key=GOOGLE_API_KEY.strip()) if GOOGLE_API_KEY else None

# 2. DeepSeek
# ----------------------------------------------------------------------
# ЧТО ИЗМЕНЕНО (v4.1): ключ DeepSeek запрашивается и обязателен только при AI_PROVIDER = "deepseek"
# ПОЧЕМУ: Т-66 — при провайдере Google ячейка падала без ключа DeepSeek, который не нужен
# ----------------------------------------------------------------------
DEEPSEEK_API_KEY = None
client_deepseek_async = client_deepseek_sync = None
if AI_PROVIDER == "deepseek":
    DEEPSEEK_API_KEY = userdata.get('DEEPSEEK_API_KEY')
    if not DEEPSEEK_API_KEY:
        raise ValueError("Секрет DEEPSEEK_API_KEY не задан в Colab (значок ключа слева).")

    client_deepseek_async = AsyncOpenAI(
        api_key=DEEPSEEK_API_KEY.strip(),
        base_url="https://api.deepseek.com/v1"
    )
    client_deepseek_sync = OpenAI(
        api_key=DEEPSEEK_API_KEY.strip(),
        base_url="https://api.deepseek.com/v1"
    )

if AI_PROVIDER == "google" and not GOOGLE_API_KEY:
    raise ValueError("Выбран Google, но ключ GEMINI_API_KEY отсутствует.")

# Совместимость: универсальный клиент под выбранный провайдер
client = client_gemini if AI_PROVIDER == "google" else client_deepseek_sync

# ----------------------------------------------------------------------
# ЧТО ИЗМЕНЕНО: безусловное drive.mount заменено функцией ensure_drive()
# ПОЧЕМУ: монтируем только по требованию — за шаблоном, которого нет в кэше,
#         или чтобы записать историю новой недели. Повторный прогон той же
#         недели проходит вообще без обращения к Диску и без окна согласия
# ----------------------------------------------------------------------
from google.colab import drive

_DRIVE_READY = {'ok': os.path.exists('/content/drive/MyDrive')}

def ensure_drive(reason=''):
    """Монтирует Диск по требованию. Возвращает True, если Диск доступен."""
    if DRIVE_MODE == "never":
        print(f"🚫 Диск отключён (DRIVE_MODE=never){': ' + reason if reason else ''}")
        return False
    if _DRIVE_READY['ok']:
        return True
    print(f"🔗 Подключаю Google Диск{': ' + reason if reason else ''}...")
    try:
        drive.mount('/content/drive', force_remount=False)
        _DRIVE_READY['ok'] = os.path.exists('/content/drive/MyDrive')
    except Exception as e:
        print(f"⚠️ Не удалось подключить Диск: {e}")
        _DRIVE_READY['ok'] = False
    if _DRIVE_READY['ok']:
        os.makedirs(TARGET_DIR, exist_ok=True)
        os.makedirs(HISTORY_DIR, exist_ok=True)
    return _DRIVE_READY['ok']

os.makedirs(CACHE_DIR, exist_ok=True)
if USE_FILE_PICKER:
    os.makedirs(TARGET_DIR, exist_ok=True)      # /content/result — локально, Диск не нужен
if DRIVE_MODE == "always":
    ensure_drive('режим always')

print(f"🤖 Активный провайдер: {AI_PROVIDER.upper()} | Батч-модель: {BATCH_MODEL} | Аналитическая: {SUMMARY_MODEL}")

# --- ЗАГРУЗКА ДАТАСЕТА (РАБОТА С ФАЙЛАМИ) ---
if USE_FILE_PICKER:
    print("📁 Выберите файлы: выгрузку (JSON/CSV), при необходимости шаблон (*.html) и history.json:")
    uploaded = files.upload()
    if not uploaded:
        raise FileNotFoundError("Файл не был выбран.")

    # ЧТО ИЗМЕНЕНО: можно выбрать несколько файлов — выгрузку, шаблон и историю
    # ПОЧЕМУ: так шаблон и история берутся без Диска, если его не хочется подключать
    file_name, file_bytes = None, None
    for _name, _bytes in uploaded.items():
        low = _name.lower()
        if low.endswith('.html'):
            with open(os.path.join(CACHE_DIR, _name), 'wb') as _f:
                _f.write(_bytes)
            print(f"🎨 Шаблон принят в кэш: {_name}")
        elif 'markup_cache' in low and low.endswith('.json'):
            # ЧТО ИЗМЕНЕНО (v4.1): принимается кэш разметки с прошлого запуска (Т-63)
            with open(os.path.join(CACHE_DIR, 'markup_cache.json'), 'wb') as _f:
                _f.write(_bytes)
            print(f"💾 Кэш разметки принят: {_name}")
        elif 'history' in low and low.endswith('.json'):
            with open(os.path.join(CACHE_DIR, 'history.json'), 'wb') as _f:
                _f.write(_bytes)
            print(f"🗓️ История недель принята в кэш: {_name}")
        elif low.endswith(('.json', '.csv')) and file_name is None:
            file_name, file_bytes = _name, _bytes

    if file_name is None:
        raise FileNotFoundError("Среди выбранных файлов нет выгрузки (.json или .csv).")
    print(f"📥 Выгрузка: {file_name}")

    if file_name.endswith('.json'):
        df = pd.read_json(io.BytesIO(file_bytes))
    else:
        df = pd.read_csv(io.BytesIO(file_bytes), sep=';', encoding='utf-8')
else:
    SOURCE_DIR = next(
        (path for path in SOURCE_DIR_VARIANTS if os.path.exists(path) and glob.glob(os.path.join(path, '*.*'))),
        None
    )
    if not SOURCE_DIR:
        raise FileNotFoundError("Исходные директории на Google Диске не найдены.")

    latest_file = max(
        glob.glob(os.path.join(SOURCE_DIR, '*.csv')) + glob.glob(os.path.join(SOURCE_DIR, '*.json')),
        key=os.path.getmtime
    )
    print(f"📄 Загрузка последнего файла с Google Диска: {latest_file}")
    df = pd.read_json(latest_file) if latest_file.endswith('.json') else pd.read_csv(latest_file, sep=';', encoding='utf-8')

# --- ПАРСИНГ ДАТ И ФИЛЬТРАЦИЯ ---
# ----------------------------------------------------------------------
# ЧТО ИЗМЕНЕНО: разбор новых полей st/res, отбор закрытых по статусу вместо наличия cls
# ПОЧЕМУ: в выгрузке теперь есть открытые и отклонённые обращения — они не должны
#         попадать в анализ качества, но нужны для новых таблиц
# ----------------------------------------------------------------------
df['reg_dt'] = pd.to_datetime(df['reg'], errors='coerce')
df['cls_dt'] = pd.to_datetime(df['cls'], errors='coerce')
df['res_dt'] = pd.to_datetime(df['res'], errors='coerce') if 'res' in df.columns else pd.NaT

df['id'] = df['id'].astype(str).str.strip()
df['st'] = df['st'].astype(str).str.strip() if 'st' in df.columns else ''
df['type'] = df['type'].astype(str).str.strip() if 'type' in df.columns else ''
df['sec'] = df['sec'].astype(str).str.strip()

# Дата завершения = дата предложенного решения; если её нет — дата подтверждения
df['done_dt'] = df['res_dt'].fillna(df['cls_dt'])

df_all = df.copy()                                                   # вся выгрузка
_closed_pool = df[df['st'].isin(CLOSED_STATUSES)].dropna(subset=['done_dt']).copy()
df_open = df[~df['st'].isin(CLOSED_STATUSES + [REJECTED_STATUS])].copy()
_rejected_pool = df[df['st'] == REJECTED_STATUS].copy()

# ----------------------------------------------------------------------
# ЧТО ИЗМЕНЕНО: введены жёсткие границы отчётной недели (пн–вс), по ним режется
#               всё, что в отчёте называется «неделей»
# ПОЧЕМУ: раньше «неделя N» была только ярлыком — в анализ попадала вся выгрузка,
#         из-за чего суточные графики и заголовок показывали разные периоды
# ----------------------------------------------------------------------
_last_done = _closed_pool['done_dt'].max()
WEEK_START = (_last_done.normalize() - pd.Timedelta(days=int(_last_done.weekday())))
WEEK_END = WEEK_START + pd.Timedelta(days=6)                # воскресенье, 00:00
WEEK_END_EXCL = WEEK_START + pd.Timedelta(days=7)           # граница «строго меньше»
_iso = WEEK_START.isocalendar()
max_year, max_week = int(_iso[0]), int(_iso[1])
WEEK_LABEL = f"Неделя {max_week} · {WEEK_START:%d.%m} — {WEEK_END:%d.%m.%Y}"
WEEK_DAYS = [WEEK_START + pd.Timedelta(days=i) for i in range(7)]

def _in_week(s):
    """Строки, чья дата попала в отчётную неделю."""
    return (s >= WEEK_START) & (s < WEEK_END_EXCL)

df_closed = _closed_pool[_in_week(_closed_pool['done_dt'])].copy()
df_rejected = _rejected_pool[_in_week(_rejected_pool['cls_dt'])].copy()

df_closed['iso_year'] = df_closed['done_dt'].dt.isocalendar().year
df_closed['iso_week'] = df_closed['done_dt'].dt.isocalendar().week

# Хвост выгрузки за пределами недели в отчёт не выводится, но считается для проверки
OUT_OF_WEEK_CLOSED = len(_closed_pool) - len(df_closed)
REJECTED_NO_DATE = int(_rejected_pool['cls_dt'].isna().sum())

# Отклонённое — тоже завершённое, но нерешённое: идёт только в счётчик
CLOSED_TOTAL = len(df_closed)
REJECTED_TOTAL = len(df_rejected)
DONE_TOTAL = CLOSED_TOTAL + REJECTED_TOTAL

# ----------------------------------------------------------------------
# ЧТО ИЗМЕНЕНО (v4.1): REPORT_DATE — дата выгрузки из имени файла (sppr_dump_ГГГГММДД_ЧЧММСС_…), now() — только если даты в имени нет
# ПОЧЕМУ: Т-65 — возраст открытых «плыл» при каждом повторном прогоне; теперь тот же файл даёт тот же отчёт
# ----------------------------------------------------------------------
_src_name = os.path.basename(file_name if USE_FILE_PICKER else latest_file)
_m_date = re.search(r'(20\d{2})(\d{2})(\d{2})_\d{6}', _src_name)
try:
    REPORT_DATE = pd.Timestamp(int(_m_date.group(1)), int(_m_date.group(2)), int(_m_date.group(3))) if _m_date else pd.Timestamp.now().normalize()
except ValueError:
    _m_date = None
    REPORT_DATE = pd.Timestamp.now().normalize()
print(f"📅 REPORT_DATE = {REPORT_DATE:%d.%m.%Y} ({'по дате выгрузки в имени файла' if _m_date else 'дата в имени файла не найдена — берём текущую'})")
df = df_closed.copy()

if TEST_SAMPLE_SIZE > 0:
    df = df.head(TEST_SAMPLE_SIZE).copy()
    print(f"\U0001f9ea ТЕСТ-РЕЖИМ: Взято {len(df)} строк. История не пишется.")
else:
    print(f"\u2702\ufe0f РАБОЧИЙ РЕЖИМ: {WEEK_LABEL}")
    print(f"   Завершено за неделю: {DONE_TOTAL} ({CLOSED_TOTAL} решено + {REJECTED_TOTAL} отклонено) | "
          f"Открытых: {len(df_open)} | Всего в выгрузке: {len(df_all)}")
    if OUT_OF_WEEK_CLOSED:
        print(f"   \u2139\ufe0f Вне недели осталось закрытых: {OUT_OF_WEEK_CLOSED} (в отчёт не входят)")
    if REJECTED_NO_DATE:
        print(f"   \u26a0\ufe0f Отклонённых без даты отклонения: {REJECTED_NO_DATE} (не попали ни в одну неделю)")

# БЛОК 2: РАСЧЕТ SLA (reg -> res, деление на 3)

In [ ]:
# =====================================================================
# [ЯЧЕЙКА 2] БЛОК 2: РАСЧЕТ SLA
# =====================================================================
# 🎛️ ГЛАВНЫЙ ПЕРЕКЛЮЧАТЕЛЬ 🎛️
ENABLE_MAIN = True #@param {type:"boolean"}

if ENABLE_MAIN:
    # -----------------------------------------------------------------
    # ЧТО ИЗМЕНЕНО: SLA = (дата предложенного решения − дата регистрации) / 3
    # ПОЧЕМУ: в сутках 8 рабочих часов, поэтому календарные часы делятся на 3.
    #         busday_count, смещение по началу смены и вычет 24 ч. больше не нужны —
    #         даты в выгрузке приходят с временем
    # -----------------------------------------------------------------
    delta_hours = (df['done_dt'] - df['reg_dt']).dt.total_seconds() / 3600.0
    df['dif_Hour'] = np.round(np.maximum(0.0, delta_hours / SLA_DIVIDER), 1)

    _zero_share = float((df['dif_Hour'] == 0).mean()) * 100 if len(df) else 0.0
    print(f"✅ SLA рассчитан (reg → res, деление на {SLA_DIVIDER}). "
          f"Медиана: {df['dif_Hour'].median():.1f} ч. | Среднее: {df['dif_Hour'].mean():.1f} ч. | "
          f"Нулевых: {_zero_share:.0f}%")
    if _zero_share > 20:
        print("⚠️ Более 20% обращений имеют SLA = 0 ч. Проверьте, что в полях reg и res есть время, а не только дата.")
else:
    print("⏭️ ПРОПУСК: Отчет (ENABLE_MAIN = False).")

# БЛОК 3 и 4: АСИНХРОННАЯ ПАКЕТНАЯ ОБРАБОТКА

In [ ]:
# =====================================================================
# [ЯЧЕЙКА 3] БЛОК 3 и 4: АСИНХРОННАЯ ПАКЕТНАЯ ОБРАБОТКА
# =====================================================================
# Здесь модель размечает КАЖДОЕ обращение по отдельности. Группировка тем
# делается отдельным вторым проходом — в ЯЧЕЙКЕ 4, по всему списку сразу.
# =====================================================================
class TicketResult(BaseModel):
    id: str = Field(description="Идентификатор тикета")
    topic_cluster: str = Field(description="Конкретная суть сбоя/операции. Без абстракций.")
    # ЧТО ИЗМЕНЕНО: добавлены issue_summary, recommendation, suggested_type, type_argument
    # ПОЧЕМУ: в отчёте нужна короткая суть вместо простыни текста, рекомендация
    #         в карточках кейсов и проверка правильности типа обращения
    issue_summary: str = Field(description="Суть обращения одной строкой, максимум 90 символов, без вводных слов")
    recommendation: str = Field(description="Что сделать, чтобы обращение не повторилось. Одно конкретное действие")
    suggested_type: str = Field(description="Тип обращения по существу текста, строго из списка допустимых типов")
    type_argument: str = Field(description="Почему тип именно такой. Заполнять только если suggested_type отличается от фактического, иначе пустая строка")
    tone_deviation_score: int = Field(description="Шкала тона от 0 до 3")
    is_emotional: bool = Field(description="True если tone_deviation_score > 0")
    emotion_argument: str = Field(description="Обоснование оценки тона")
    complexity: int = Field(description="Сложность задачи от 1 до 5")
    initial_grade: str = Field(description="Исходная оценка")
    final_grade: str = Field(description="Итоговый грейд: ПЛОХО, ХОРОШО, ОТЛИЧНО")
    grade_argument: str = Field(description="Детальная аргументация грейда и причин снижения")
    is_alternative: bool = Field(description="True если ИИ не согласен с закрытием тикета")
    is_critical_incident: bool = Field(description="True если инцидент блокирует бизнес-процесс")
    planshet: bool = Field(description="True только если в тексте есть слово планшет или его форма")
    planshet_argument: str = Field(description="Цитата со словом планшет. Пустая строка если planshet=false")

class BatchResponse(BaseModel):
    results: list[TicketResult]

if ENABLE_MAIN:
    # ----------------------------------------------------------------------
    # ЧТО ИЗМЕНЕНО: критерии грейда ужесточены, добавлены правила для
    #               issue_summary, recommendation и проверки типа обращения
    # ПОЧЕМУ: оценка качества была слишком мягкой — почти всё уходило в ХОРОШО
    # ----------------------------------------------------------------------
    SYSTEM_INSTRUCTION = f"""
    Вы — ведущий системный аналитик и архитектор 1С. Ваша задача — провести жесткий, технически объективный аудит пакета обращений в службу поддержки.
    Вы оцениваете строго. Оценка по умолчанию — ХОРОШО, и она ПОВЫШАЕТСЯ только за доказанную превентивность и ПОНИЖАЕТСЯ при любом из признаков ниже. Не завышайте оценки из вежливости.

    0. ЖЁСТКИЕ ЛИМИТЫ ДЛИНЫ — соблюдайте строго, ответ должен уместиться целиком:
    - issue_summary — до 90 символов, recommendation — до 120, grade_argument — до 250,
      emotion_argument — до 120, type_argument — до 120, topic_cluster — до 60.
    - Пишите плотно, без вводных оборотов и повторов текста обращения.


    1. ПРАВИЛА КЛАСТЕРИЗАЦИИ (topic_cluster):
    - Категорически запрещены абстрактные формулировки: "Ошибка системы", "Проблема с 1С", "Консультация", "Вопрос по ДО".
    - Указывайте КОНКРЕТНУЮ суть сбоя или операции. Пример: "Ошибка при проведении документа Заказ", "Неверный расчет лимита БТД", "Зависание сессии при экспорте".
    - Формулируйте кратко и единообразно: отглагольное существительное + объект. "Создание контрагента", "Изменение подписанта договора", "Пересчёт лимита БТД".
    - Не добавляйте в формулировку слова "в 1С", "в базе 1С", "в системе" — они не несут смысла и плодят синонимы одной темы.

    1.1. СУТЬ ОБРАЩЕНИЯ (issue_summary):
    - Одна строка до 90 символов: что именно не работало или что просили сделать.
    - Без вводных слов "пользователь сообщает", "обращение о том, что". Сразу факт.
    - Не копируйте текст обращения — формулируйте своими словами. Пример: "Не создаётся проект ДС №2 при наличии ДС №1".

    1.2. РЕКОМЕНДАЦИЯ (recommendation):
    - Одно конкретное действие, которое исключит повтор: доработка, изменение настройки, инструкция в базу знаний, автоматическая проверка.
    - Не пишите "усилить контроль", "повысить внимательность" — только проверяемое действие.

    2. КРИТЕРИИ ОЦЕНКИ ГРЕЙДА (final_grade) — ОЦЕНИВАЙТЕ СТРОГО:
    - ПЛОХО (ставьте при ЛЮБОМ из признаков):
      * Решение содержит прямые ручные правки в базе данных (SQL/1С) без устранения причины.
      * Причина сбоя не названа: написано "исправлено", "устранено", "восстановлено" без объяснения, что именно было не так.
      * Аналитик перенаправил тикет без ответа или закрыл с формулировкой "дубликат/ошибка" без объяснений.
      * Ответ не решает проблему пользователя, формальная отписка.
      * Время решения несоразмерно задаче (простая консультация висела несколько дней).
      * Проблема повторяется у разных пользователей, но решается точечно, без заявки на доработку.
      * Удаление ЭЦП если нет согласования пользователя.
      * Текст решения пустой, состоит из одного слова или из даты без содержания.
    - ХОРОШО:
      * Стандартное корректное решение: названа причина, описано действие, проблема закрыта.
      * Удаление ЭЦП если есть согласование пользователя.
      * Обращения с темой "Закрытие отчетного периода", "Встречи", "Планерка" являются ХОРОШО.
      * Обращения содержащие примерный текст "Ваше обращение передано в Сервис Деск. Запрос на обслуживание номер" - это норма, значит не наше обращение.
    - ОТЛИЧНО (ставьте ТОЛЬКО при доказанной превентивности, не более чем каждому пятому обращению):
      * Аналитик не просто исправил ошибку, но и завёл задачу на баг-трекер с номером, предложил автоматизацию или дал инструкцию, исключающую повторение.
      * Одного лишь подробного ответа для ОТЛИЧНО НЕДОСТАТОЧНО — нужен след, который уменьшит поток обращений в будущем.

    2.1. ПРОВЕРКА ТИПА ОБРАЩЕНИЯ (suggested_type):
    - Допустимые значения строго из списка: {', '.join(KNOWN_TYPES)}.
    - Определяйте тип ПО СУЩЕСТВУ ОБРАЩЕНИЯ, не по тому, что указано в поле "Тип (как указан)".
    - РЕШЕНИЕ ВАЖНЕЕ ФОРМУЛИРОВКИ ОБРАЩЕНИЯ. Инициатор часто описывает собственное
      непонимание как поломку системы. Тип определяется тем, ЧТО РЕАЛЬНО ПРИШЛОСЬ СДЕЛАТЬ:
      * решение — объяснение, как пользоваться, куда нажать, почему так задумано, указание
        на ошибку самого пользователя -> Консультация, даже если в описании "ошибка",
        "не работает", "сбой", "не могу";
      * решение — правка, заведение или удаление данных -> Изменение данных в системе;
      * решение — выдача, снятие или делегирование прав -> Права доступа;
      * решение — исправление кода, настройки, перезапуск, передача в разработку,
        установка релиза -> Инцидент;
      * решение — ссылка на ранее созданное обращение -> Дубль.
    - Если поле "Решение" пустое или из него не видно, что именно делали, — определяйте
      тип по описанию.
    - Ориентиры по описанию (когда решение не помогает):
      * Инцидент — что-то работавшее перестало работать, ошибка, сбой, блокировка процесса.
      * Изменение данных в системе — просьба поправить/завести/удалить данные, когда система работает штатно.
      * Консультация — вопрос "как сделать", методическая помощь, разъяснение, обучение.
      * Права доступа — выдача, снятие, делегирование прав и учётных записей.
      * Дубль — повтор ранее созданного обращения.
    - Если тип по существу совпадает с указанным — type_argument оставьте ПУСТЫМ.
    - Если отличается — в type_argument коротко объясните, на что вы опирались:
      на решение или на описание, и по какому признаку.

    3. ШКАЛА ЭМОЦИОНАЛЬНОГО ТОНА (tone_deviation_score):
    - 0 (Деловой): Конструктивное описание проблемы, стандартный рабочий диалог.
    - 1 (Скрытое недовольство): Ирония, сарказм, жалобы на длительность ("Сколько можно ждать", "Опять ничего не работает").
    - 2 (Явная агрессия): Требования эскалации, капслок в претензиях, угрозы жалоб руководству.
    - 3 (Оскорбления): Прямая ругань, переходы на личности.

    ИСКЛЮЧЕНИЯ (ОБЯЗАТЕЛЬНО СТАВИТЬ ТОН 0):
    - Множественные знаки препинания в приветствиях или эмодзи (например: "Добрый день!!!", "Спасибо!").
    - Написание аббревиатур и названий систем CAPS LOCK (например: "Ошибки в ЗУП", "Не проводится ГПД").
    - Использование слова "СРОЧНО" или "КРИТИЧНО", если бизнес-процесс действительно заблокирован.

    4. СЛОЖНОСТЬ (complexity):
    - Оценка от 1 (простая консультация/кнопка) до 5 (комплексный сбой архитектуры/перерасчет регистров).

    5. АЛЬТЕРНАТИВНОЕ МНЕНИЕ (is_alternative):
    - Ставьте true, если первоначальный статус тикета или ответ аналитика формально закрывает обращение, но по существу проблема пользователя НЕ решена, или решение содержит критическую ошибку/риск для системы.

    6. КРИТИЧЕСКИЙ ИНЦИДЕНТ (is_critical_incident):
    - Ставьте true, если сбой заблокировал работу подразделения, вызвал остановку отгрузок/расчета зарплаты или затронул более 5 пользователей одновременно.

    7. ПРИЗНАК ПЛАНШЕТА (planshet) — ПО УМОЛЧАНИЮ false:
    - true ТОЛЬКО если в тексте темы, описания или решения встречается слово "планшет"
      или его форма: планшета, планшете, планшетом, планшеты, планшетов, планшетный, планшетная.
    - Никакие другие устройства и приложения признаком планшета НЕ являются: ни ТСД, ни мобильное
      приложение, ни мобильный АРМ, ни Android, ни iPad, ни смартфон, ни ноутбук.
    - НЕ ДОДУМЫВАЙТЕ и не обобщайте. Нет слова "планшет" в тексте — planshet = false.
    - planshet_argument: при true — дословная цитата из текста со словом "планшет"; при false — пустая строка.

    Перед вынесением итоговых оценок ОБЯЗАТЕЛЬНО заполняйте аргументацию (emotion_argument, grade_argument), подробно объясняя причину снижения грейда или фиксации эмоционального отклонения.
    """

    async def call_ai_provider_batch(prompt: str):
        """Универсальный вызов для Google и DeepSeek"""
        if AI_PROVIDER == "google":
            response = await client_gemini.aio.models.generate_content(
                model=BATCH_MODEL, contents=prompt,
                config={
                    'system_instruction': SYSTEM_INSTRUCTION,
                    'response_mime_type': 'application/json',
                    'response_schema': BatchResponse,
                    'temperature': 0.1,
                    'max_output_tokens': MAX_OUTPUT_TOKENS
                }
            )
            tokens = {
                'prompt': response.usage_metadata.prompt_token_count if response.usage_metadata else 0,
                'candidates': response.usage_metadata.candidates_token_count if response.usage_metadata else 0,
                'total': response.usage_metadata.total_token_count if response.usage_metadata else 0
            }
            data = json.loads(response.text).get('results', [])
            return data, tokens

        else: # deepseek
            schema_format = BatchResponse.model_json_schema()
            full_system = f"{SYSTEM_INSTRUCTION}\nОтвет строго в формате JSON по схеме:\n{json.dumps(schema_format, ensure_ascii=False)}"

            response = await client_deepseek_async.chat.completions.create(
                model=BATCH_MODEL,
                messages=[
                    {"role": "system", "content": full_system},
                    {"role": "user", "content": prompt}
                ],
                response_format={"type": "json_object"},
                temperature=0.1,
                max_tokens=MAX_OUTPUT_TOKENS
            )
            usage = response.usage
            tokens = {
                'prompt': usage.prompt_tokens if usage else 0,
                'candidates': usage.completion_tokens if usage else 0,
                'total': usage.total_tokens if usage else 0
            }
            res_raw = json.loads(response.choices[0].message.content)
            data = res_raw.get('results', []) if isinstance(res_raw, dict) else res_raw
            return data, tokens

    # ----------------------------------------------------------------------
    # ЧТО ИЗМЕНЕНО (v4.1): заглушка при сбое ИИ ставит оценку «Н/Д», а не «ХОРОШО»
    # ПОЧЕМУ: Т-64 — сбой модели был неотличим от нормальной оценки и попадал в рейтинг как «хорошо»
    # ----------------------------------------------------------------------
    def _fallback_rows(chunk, err):
        """Заглушки по каждой строке пачки, если разметка не удалась."""
        return [
            {
                'id': str(r['id']),
                'topic_cluster': str(r.get('sec', 'Не определено')).strip() or 'Без темы',
                'issue_summary': str(r.get('sec', '')).strip() or 'Нет данных',
                'recommendation': '', 'suggested_type': str(r.get('type', '')).strip(), 'type_argument': '',
                'tone_deviation_score': 0, 'is_emotional': False, 'emotion_argument': '',
                'complexity': 1, 'initial_grade': 'Н/Д', 'final_grade': 'Н/Д',
                'grade_argument': f'Ошибка вызова API ({AI_PROVIDER}): {err}',
                'is_alternative': False, 'is_critical_incident': False,
                'planshet': False, 'planshet_argument': ''
            } for _, r in chunk.iterrows()
        ]

    # ----------------------------------------------------------------------
    # ЧТО ИЗМЕНЕНО (v4.1): кэш разметки по id — готовая разметка обращения берётся из кэша, а не покупается заново
    # ПОЧЕМУ: Т-63 — повторный прогон той же выгрузки оплачивал токены на все обращения. Ключ кэша = тексты, которые
    #         видит модель + промпт + модель: правка решения, промпта или смена модели сбрасывает кэш по обращению сама.
    #         Сбойные заглушки в кэш не попадают — при следующем запуске они разметятся заново.
    # ----------------------------------------------------------------------
    import hashlib
    MARKUP_CACHE_FILE = os.path.join(CACHE_DIR, 'markup_cache.json')
    MARKUP_CACHE_DRIVE = os.path.join(HISTORY_DIR, 'markup_cache.json')

    def _markup_key(r):
        parts = [str(r.get(k, '') or '') for k in ('cli', 'anl', 'sec', 'type')]
        parts += [str(r.get(k, '') or '')[:PROMPT_TEXT_LIMIT] for k in ('desc', 'sol')]
        parts += [SYSTEM_INSTRUCTION, BATCH_MODEL]
        return hashlib.sha1('\x1f'.join(parts).encode('utf-8')).hexdigest()

    def load_markup_cache():
        paths = [MARKUP_CACHE_FILE]
        if DRIVE_MODE == "always" and ensure_drive('кэш разметки'):
            paths.append(MARKUP_CACHE_DRIVE)
        cache = {}
        for p in paths:
            if os.path.exists(p):
                try:
                    with open(p, 'r', encoding='utf-8') as f:
                        cache.update(json.load(f))
                except Exception as e:
                    print(f"⚠️ Кэш разметки {p} не прочитан ({e}) — игнорируем")
        return cache

    def save_markup_cache(cache):
        os.makedirs(CACHE_DIR, exist_ok=True)
        with open(MARKUP_CACHE_FILE, 'w', encoding='utf-8') as f:
            json.dump(cache, f, ensure_ascii=False)
        if DRIVE_MODE == "always" and ensure_drive('запись кэша разметки'):
            os.makedirs(HISTORY_DIR, exist_ok=True)
            with open(MARKUP_CACHE_DRIVE, 'w', encoding='utf-8') as f:
                json.dump(cache, f, ensure_ascii=False)

    async def process_chunk(chunk_id, chunk, sem):
        """Одна пачка. Возвращает (строки, токены, ok). ok=False — пачку надо переразметить."""
        prompt_rows = []
        for _, r in chunk.iterrows():
            # ЧТО ИЗМЕНЕНО: в промпт добавлен фактический тип обращения, тексты обрезаются
            # ПОЧЕМУ: без типа модель не увидит неверную классификацию; длинные тексты
            #         раздувают вход и провоцируют обрыв ответа
            prompt_rows.append(
                f"ID: {r['id']} | Инициатор: {r.get('cli','')} | Аналитик: {r.get('anl','')}\n"
                f"Тема: {r.get('sec','')}\n"
                f"Тип (как указан): {r.get('type','')}\n"
                f"Описание: {str(r.get('desc','') or '')[:PROMPT_TEXT_LIMIT]}\n"
                f"Решение: {str(r.get('sol','') or '')[:PROMPT_TEXT_LIMIT]}\n"
                f"----------------------------------------"
            )
        prompt = "Оцени следующий пакет обращений:\n" + "\n".join(prompt_rows)

        max_retries = 3
        delay = SLEEP_TIME
        for attempt in range(1, max_retries + 1):
            try:
                async with sem:
                    await asyncio.sleep(delay)
                    start_t = time.time()
                    data, tokens = await call_ai_provider_batch(prompt)
                print(f"✅ Пакет {chunk_id} завершен ({time.time() - start_t:.1f}s, строк: {len(chunk)}, токенов: {tokens['total']})")
                return data, tokens, True

            except json.JSONDecodeError as e:
                # ---------------------------------------------------------
                # ЧТО ИЗМЕНЕНО: обрыв JSON больше НЕ заполняется заглушками
                # ПОЧЕМУ: обрезанный ответ — это потеря данных; пачка возвращается
                #         как несостоявшаяся и переразмечается меньшим размером
                # ---------------------------------------------------------
                print(f"✂️ Пакет {chunk_id}: ответ обрезан на позиции {e.pos} — пачку из {len(chunk)} строк отправляем на переразметку")
                return [], {'prompt': 0, 'candidates': 0, 'total': 0}, False

            except Exception as e:
                print(f"⚠️ Сбой пакета {chunk_id} (попытка {attempt}/{max_retries}): {e}")
                if attempt == max_retries:
                    return [], {'prompt': 0, 'candidates': 0, 'total': 0}, False
                delay *= 2

    async def run_pass(frame, size, pass_no):
        """Один проход по строкам frame пачками по size. Возвращает (строки, токены, id_на_повтор)."""
        sem = asyncio.Semaphore(MAX_CONCURRENCY)
        chunks = [frame.iloc[i:i + size] for i in range(0, len(frame), size)]
        print(f"🚀 Проход {pass_no}: строк {len(frame)}, пачек {len(chunks)} по {size}, потоков {MAX_CONCURRENCY}")

        results = await asyncio.gather(*[process_chunk(f"{pass_no}.{i}", c, sem) for i, c in enumerate(chunks, 1)])

        flat, toks, retry_ids = [], {'prompt': 0, 'candidates': 0, 'total': 0}, []
        for idx, ((data, t, ok), chunk) in enumerate(zip(results, chunks), 1):
            for k in toks:
                toks[k] += t.get(k, 0)
            chunk_ids = set(chunk['id'].astype(str))
            if not ok:
                retry_ids.extend(chunk_ids)
                continue
            got = set()
            for row in data:
                rid = str(row.get('id', '')).strip()
                if rid in chunk_ids and rid not in got:
                    flat.append(row)
                    got.add(rid)
            # ЧТО ИЗМЕНЕНО: строки, которых модель не вернула, тоже идут на повтор
            # ПОЧЕМУ: раньше они молча оставались без оценки ИИ
            missing = chunk_ids - got
            if missing:
                print(f"↩️ Пакет {pass_no}.{idx}: модель не вернула {len(missing)} строк — на повтор")
                retry_ids.extend(missing)
        return flat, toks, retry_ids

    async def run_batching():
        """Проходы с уменьшением пачки: сбойные строки переразмечаются, а не заглушаются."""
        size = BATCH_SIZE
        cache = load_markup_cache()
        key_by_id = {str(r['id']): _markup_key(r) for _, r in df.iterrows()}
        cached_rows, todo = [], []
        for idx, r in df.iterrows():
            hit = cache.get(str(r['id']))
            if hit and hit.get('key') == key_by_id[str(r['id'])]:
                cached_rows.append(hit['row'])
            else:
                todo.append(idx)
        pending = df.loc[todo].copy()
        print(f"💾 Кэш разметки: из кэша {len(cached_rows)}, к разметке ИИ {len(pending)} из {len(df)}")
        flat_results = []
        total_batch_tokens = {'prompt': 0, 'candidates': 0, 'total': 0}
        start_total = time.time()
        pass_no = 0

        while len(pending) and pass_no < 6:
            pass_no += 1
            data, toks, retry_ids = await run_pass(pending, size, pass_no)
            flat_results.extend(data)
            for k in total_batch_tokens:
                total_batch_tokens[k] += toks[k]

            if not retry_ids:
                pending = pending.iloc[0:0]
                break

            pending = pending[pending['id'].astype(str).isin(set(retry_ids))]
            if size <= 1:
                break
            size = max(1, size // 2)
            print(f"♻️ На повтор {len(pending)} строк. Уменьшаем пачку до {size} и размечаем заново.")

        # новая разметка — в кэш (до добавления заглушек: сбои не кэшируются)
        for row in flat_results:
            rid = str(row.get('id', '')).strip()
            if rid in key_by_id:
                cache[rid] = {'key': key_by_id[rid], 'row': row}
        if flat_results:
            save_markup_cache(cache)
            print(f"💾 Кэш разметки обновлён: {len(cache)} записей → {MARKUP_CACHE_FILE}" + (" (скачайте файл, чтобы приложить его при следующем запуске)" if USE_FILE_PICKER else ""))

        if len(pending):
            print(f"⛔ {len(pending)} строк не удалось разметить даже по одной. Они помечены как «Н/Д» (ошибка разметки).")
            flat_results.extend(_fallback_rows(pending, 'ответ модели не разобран'))

        flat_results = cached_rows + flat_results
        print(f"✅ Разметка завершена за {time.time() - start_total:.1f} сек. "
              f"Строк: {len(flat_results)} из {len(df)} (из кэша {len(cached_rows)}) | токенов: {total_batch_tokens['total']}")
        return flat_results, total_batch_tokens

    # Запуск
    ai_results, total_batch_tokens = await run_batching()

    # -----------------------------------------------------------------
    # ЧТО ИЗМЕНЕНО: перед merge обе стороны приводятся к строковому id,
    #               дубликаты ответов модели схлопываются
    # ПОЧЕМУ: несовпадение типов ключа (int в CSV против str в схеме) давало
    #         пустое слияние — все AI-поля NaN. Повтор id размножал строки
    # -----------------------------------------------------------------
    df_ai = pd.DataFrame(ai_results)
    if not df_ai.empty:
        df_ai['id'] = df_ai['id'].astype(str).str.strip()
        df_ai = df_ai.drop_duplicates(subset=['id'], keep='first')
    df_final = pd.merge(df, df_ai, on='id', how='left')

    _no_grade = int(df_final['final_grade'].isna().sum()) if 'final_grade' in df_final.columns else len(df_final)
    print(f"🔗 Слияние: {len(df_final)} строк | без оценки ИИ: {_no_grade}")
    if _no_grade:
        print("⚠️ Часть строк осталась без оценки ИИ — проверьте совпадение id в выгрузке и в ответе модели.")
else:
    print("⏭️ ПРОПУСК: Отчет (ENABLE_MAIN = False).")

# БЛОК 4.5: ВТОРИЧНАЯ ГРУППИРОВКА ТЕМ

In [ ]:
# =====================================================================
# [ЯЧЕЙКА 4] БЛОК 4.5: ВТОРИЧНАЯ ГРУППИРОВКА ТЕМ (ПО КОНКРЕТНЫМ ОБЪЕКТАМ 1С)
# =====================================================================
import json
import asyncio

if ENABLE_MAIN and ENABLE_TOPIC_GROUPING and 'df_final' in locals() and not df_final.empty:
    print(f"\n🧩 Запуск точечной группировки тем (Объекты/Операции 1С) | Модель: {BATCH_MODEL}...")

    # 1. Извлекаем уникальные темы
    unique_topics_raw = df_final['topic_cluster'].dropna().astype(str).str.strip().unique().tolist()
    unique_topics = [t for t in unique_topics_raw if t and t.lower() != 'nan']

    if len(unique_topics) <= TOPIC_GROUPS_MIN:
        print(f"ℹ️ Уникальных тем всего {len(unique_topics)} (<= {TOPIC_GROUPS_MIN}). Группировка пропускается.")
        df_final['topic_group'] = df_final['topic_cluster']
    else:
        # Предварительная очистка названий тем от символов, ломающих JSON
        cleaned_topics = [
            t.replace('"', "'").replace('\\', '/').replace('\n', ' ').replace('\r', '')
            for t in unique_topics
        ]

        # Нумерованный список для модели
        topics_list_str = "\n".join([f"{idx + 1}. {topic}" for idx, topic in enumerate(cleaned_topics)])

        # Системный промпт
        GROUPING_SYSTEM = f"""
Вы — ведущий системный аналитик и архитектор 1С. Вам дан полный список формулировок тем обращений за одну неделю.
Ваша задача — сгруппировать их по КРАЙНЕ УЗКОЙ И КОНКРЕТНОЙ СУТИ (до объекта системы или точечной операции).

ЖЁСТКИЕ ПРАВИЛА И ЗАПРЕТЫ:

1. КАТЕГОРИЧЕСКИ ЗАПРЕЩЕНЫ обобщающие бизнес-категории и функциональные блоки!
   УБЕРИТЕ категории вида:
   ❌ "Кадры, пропуска и обучение"
   ❌ "Согласование документов и распоряжения"
   ❌ "Договоры и допсоглашения"
   ❌ "Закупки и снабжение"
   ❌ "Учетные записи и права доступа"

2. ГРУППИРУЙТЕ СТРОГО ПО КОНКРЕТНОМУ ОБЪЕКТУ ИЛИ ТОЧЕЧНОЙ ОПЕРАЦИИ (Документ / Справочник / Обработка / Регистр / Процесс):
   ФОРМУЛИРУЙТЕ ТАК:
   ✅ "Сбой публикации тестов в Корпоративном Университете" (вместо "Обучение")
   ✅ "Ошибки проведения заказа поставщику при превышении лимитов БТД" (вместо "Закупки")
   ✅ "Неверный маршрут согласования служебных договоров" (вместо "Согласование")
   ✅ "Снятие/добавление ЭЦП в документах по запросу" (вместо "ЭДО")
   ✅ "Ошибка выгрузки/заведения прокси-карт и пропусков в ЗУП" (вместо "Кадры")
   ✅ "Зависание/ошибки входа 1С на мобильных клиентах (планшетах)" (вместо "Общетехнические")

3. ПРИНЦИП ГРУППИРОВКИ:
   - Объединяйте формулировки ТОЛЬКО если они касаются ОДНОГО И ТОГО ЖЕ объекта 1С, реквизита или конкретной ошибки/алгоритма.
   - Если две темы относятся к одному модулю, но к разным документам/справочникам — это РАЗНЫЕ группы.
   - Название группы должно давать руководителю сразу готовый диагноз (что именно сломалось/зависло/требует ручных правок).

4. ЦЕЛЕВОЕ ЧИСЛО ГРУПП: от {TOPIC_GROUPS_MIN} до {TOPIC_GROUPS_MAX}.
   Формулируйте наименования групп емко (3-6 слов).

5. КАЖДЫЙ номер из входного списка должен попасть РОВНО В ОДНУ группу.
   Если тема единичная и узкая — объединяйте с ближайшей по конкретному объекту.

6. ЗАПРЕЩЁННЫЕ НАЗВАНИЯ ГРУПП — за них ответ считается неверным:
   ❌ "Прочее", "Прочие точечные запросы", "Разное", "Другое", "Общие вопросы" — свалки запрещены полностью.
   ❌ Названия через запятую или союз "и", перечисляющие разные сущности:
      "Договоры: реквизиты и корректировки", "Кадры и пропуска", "Доступы и права".
      Если тянет написать "и" — значит вы смешали две группы, разделите их.
   ❌ Название из одного слова-модуля: "Договоры", "Закупки", "Персонал", "ЭДО".
   ✅ Правильное название = ОБЪЕКТ + ЧТО С НИМ НЕ ТАК (или какое действие):
      "Не создаётся проект ДС при действующем ДС", "Зависание визы в маршруте согласования",
      "Битые ссылки в документе Внутреннее потребление", "Печать этикетки СИЗ по неверной номенклатуре".

ФОРМАТ ОТВЕТА — строго JSON, без markdown и пояснений:
{{"groups": [{{"name": "Конкретная проблема/объект", "items": [1, 5, 12]}}]}}
В items — ТОЛЬКО номера из входного списка.
"""

        user_prompt = f"Сгруппируй следующие {len(cleaned_topics)} тем по объектам/операциям 1С:\n{topics_list_str}"

        async def fetch_topic_groups():
            for attempt in range(1, TOPIC_GROUPING_RETRIES + 1):
                try:
                    if AI_PROVIDER == "google":
                        res = await asyncio.to_thread(
                            client_gemini.models.generate_content,
                            model=BATCH_MODEL,
                            contents=f"{GROUPING_SYSTEM}\n\n{user_prompt}",
                            config={
                                'temperature': 0.1,
                                'max_output_tokens': 8192,
                                'response_mime_type': 'application/json'
                            }
                        )
                        raw_text = res.text
                    else:
                        res = await client_deepseek_async.chat.completions.create(
                            model=BATCH_MODEL,
                            messages=[
                                {"role": "system", "content": GROUPING_SYSTEM},
                                {"role": "user", "content": user_prompt}
                            ],
                            temperature=0.1,
                            max_tokens=8192,
                            response_format={"type": "json_object"}
                        )
                        raw_text = res.choices[0].message.content

                    clean_json = raw_text.replace('```json', '').replace('```', '').strip()
                    parsed = json.loads(clean_json)

                    if "groups" in parsed and isinstance(parsed["groups"], list):
                        return parsed["groups"]

                except json.JSONDecodeError as e:
                    print(f"⚠️ Сбой парсинга JSON пакета 4 (Попытка {attempt}/{TOPIC_GROUPING_RETRIES}): {e.msg} на символе {e.pos}")
                    if 'raw_text' in locals() and len(raw_text) > 0:
                        print("✂️ Хвост полученного ответа от ИИ (проверка обрыва):")
                        print(raw_text[-200:])
                except Exception as e:
                    print(f"⚠️ Сбой выполнения запроса группировки (Попытка {attempt}/{TOPIC_GROUPING_RETRIES}): {e}")

                await asyncio.sleep(2)
            return None

        # ИСПРАВЛЕНИЕ: Прямой await в ячейке Colab без вызова loop.run_until_complete()
        groups_result = await fetch_topic_groups()

        # ------------------------------------------------------------------
        # ЧТО ИЗМЕНЕНО: добавлена проверка названий групп по стоп-листу
        #               и второй проход для переформулировки плохих названий
        # ПОЧЕМУ: модель всё равно выдаёт "Прочие точечные запросы" и
        #         названия вида "Договоры: реквизиты и корректировки"
        # ------------------------------------------------------------------
        BAD_NAME_MARKERS = ['прочее', 'прочие', 'разное', 'другое', 'общие вопросы', 'иные']

        def is_bad_group_name(name):
            low = str(name).strip().lower()
            if not low:
                return True
            if any(m in low for m in BAD_NAME_MARKERS):
                return True
            if ' и ' in low or low.count(',') >= 1:     # перечисление сущностей
                return True
            if len(low.split()) <= 1:                   # одно слово-модуль
                return True
            return False

        async def refine_group_names(bad_groups):
            """Второй проход: просим переформулировать только плохие названия."""
            listing = "\n".join([
                f"{i + 1}. Текущее название: {g['name']}\n   Темы внутри: {'; '.join(g['topics'][:8])}"
                for i, g in enumerate(bad_groups)
            ])
            prompt = (
                "Названия групп ниже нарушают правила: это свалки, перечисления через запятую или союз 'и', "
                "либо одно слово-модуль. Переформулируйте КАЖДОЕ по схеме ОБЪЕКТ + ЧТО С НИМ НЕ ТАК, "
                "опираясь на темы внутри группы. Если внутри группы смешаны разные сущности — выберите "
                "доминирующую и назовите по ней.\n\n" + listing +
                '\n\nОтвет строго JSON без markdown: {"names": ["новое название 1", "новое название 2"]}'
            )
            try:
                if AI_PROVIDER == "google":
                    res = await asyncio.to_thread(
                        client_gemini.models.generate_content,
                        model=BATCH_MODEL, contents=prompt,
                        config={'temperature': 0.1, 'response_mime_type': 'application/json'})
                    raw = res.text
                else:
                    res = await client_deepseek_async.chat.completions.create(
                        model=BATCH_MODEL,
                        messages=[{"role": "user", "content": prompt}],
                        temperature=0.1, response_format={"type": "json_object"})
                    raw = res.choices[0].message.content
                names = json.loads(raw.replace('```json', '').replace('```', '').strip()).get('names', [])
                return [str(n).strip() for n in names]
            except Exception as e:
                print(f"⚠️ Второй проход по названиям групп не удался: {e}")
                return []

        # Маппинг полученных групп обратно в DataFrame
        if groups_result:
            # Сначала собираем состав групп, чтобы было что показать при переформулировке
            prepared = []
            for group in groups_result:
                g_name = str(group.get("name", "")).strip()
                idxs = [i for i in group.get("items", []) if isinstance(i, (int, str))]
                topics_in = []
                for idx in idxs:
                    try:
                        idx = int(idx)
                    except (TypeError, ValueError):
                        continue
                    if 1 <= idx <= len(unique_topics):
                        topics_in.append(unique_topics[idx - 1])
                if topics_in:
                    prepared.append({'name': g_name, 'topics': topics_in})

            bad = [g for g in prepared if is_bad_group_name(g['name'])]
            if bad:
                print(f"♻️ Названий групп на переформулировку: {len(bad)} из {len(prepared)}")
                new_names = await refine_group_names(bad)
                for g, new_name in zip(bad, new_names):
                    if new_name and not is_bad_group_name(new_name):
                        g['name'] = new_name
                still_bad = [g['name'] for g in prepared if is_bad_group_name(g['name'])]
                if still_bad:
                    print(f"⚠️ Остались неудачные названия ({len(still_bad)}): {', '.join(still_bad[:5])}")

            topic_to_group_map = {}
            for g in prepared:
                for t in g['topics']:
                    topic_to_group_map[t] = g['name'] or t

            # Применяем группу к DataFrame
            df_final['topic_group'] = df_final['topic_cluster'].map(topic_to_group_map).fillna(df_final['topic_cluster'])
            mapped_count = len(df_final['topic_group'].unique())
            print(f"✅ Успешно сформировано {mapped_count} точечных проблемных групп по объектам 1С.")
        else:
            print("⚠️ Не удалось получить валидные группы от ИИ. Использованы исходные кластеры.")
            df_final['topic_group'] = df_final['topic_cluster']
else:
    print("⏭️ ПРОПУСК: Вторичная группировка тем отключена или нет данных.")

# БЛОК 5: АГРЕГАЦИЯ И ИСТОРИЯ

In [ ]:
# =====================================================================
# [ЯЧЕЙКА 5] БЛОК 5: АГРЕГАЦИЯ И ИСТОРИЯ
# =====================================================================
if ENABLE_MAIN:
    # ----------------------------------------------------------------------
    # ЧТО ИЗМЕНЕНО (v4.1): пустая оценка → «Н/Д», а не «ХОРОШО»; число таких обращений выводится в лог
    # ПОЧЕМУ: Т-64 — обращения без оценки ИИ не должны выглядеть хорошими; в баллы рейтинга, «Плохо» и «Лучшие» они не входят
    # ----------------------------------------------------------------------
    df_final['final_grade'] = df_final.get('final_grade', pd.Series(['Н/Д']*len(df_final), index=df_final.index)).fillna('Н/Д').astype(str).str.upper().str.strip()
    _nd_grade = int((df_final['final_grade'] == 'Н/Д').sum())
    if _nd_grade:
        print(f"⚠️ Без оценки ИИ (Н/Д): {_nd_grade} из {len(df_final)} — в рейтинг, «Плохо» и «Лучшие» не входят; повторный прогон разметит их заново.")
    for col in ['complexity', 'tone_deviation_score']: df_final[col] = pd.to_numeric(df_final.get(col, 1), errors='coerce').fillna(1 if col=='complexity' else 0)

    _BOOL_MAP = {True: True, 'True': True, 'true': True, 'ИСТИНА': True, 1: True, '1': True,
                 False: False, 'False': False, 'false': False, 'ЛОЖЬ': False, 0: False, '0': False}
    for col in ['is_critical_incident', 'is_alternative', 'planshet']:
        _src = df_final[col] if col in df_final.columns else pd.Series(False, index=df_final.index)
        df_final[col] = _src.map(_BOOL_MAP).fillna(False).astype(bool)

    # Страховка на случай, если ЯЧЕЙКА 4 не выполнялась
    if 'topic_group' not in df_final.columns:
        df_final['topic_group'] = df_final.get('topic_cluster', 'Без темы')
    df_final['topic_group'] = df_final['topic_group'].astype(str).str.strip().replace('', 'Прочее')
    if 'planshet_argument' not in df_final.columns:
        df_final['planshet_argument'] = ''

    # ЧТО ИЗМЕНЕНО: страховка по новым полям схемы
    # ПОЧЕМУ: при fallback-ответе модели их может не быть, а слайды на них опираются
    for _col, _default in [('issue_summary', ''), ('recommendation', ''),
                           ('suggested_type', ''), ('type_argument', '')]:
        if _col not in df_final.columns:
            df_final[_col] = _default
        df_final[_col] = df_final[_col].fillna(_default).astype(str).str.strip()
    # Суть пустая — подставляем кластер, чтобы в таблицах не было дыр
    df_final.loc[df_final['issue_summary'] == '', 'issue_summary'] = df_final['topic_cluster']

    # ----------------------------------------------------------------------
    # ЧТО ИЗМЕНЕНО: planshet перепроверяется по тексту, а не только по ответу ИИ
    # ПОЧЕМУ: признак ставится ТОЛЬКО по слову "планшет" и его формам;
    #         модель иногда relaxes правило и помечает ТСД, Android, мобильные АРМ
    # ----------------------------------------------------------------------
    _pl_text = (df_final['sec'].astype(str) + ' ' + df_final['desc'].astype(str) + ' ' + df_final['sol'].astype(str)).str.lower()
    _pl_hit = _pl_text.str.contains('планшет', na=False)
    _pl_dropped = int((df_final['planshet'] & ~_pl_hit).sum())
    df_final['planshet'] = df_final['planshet'] & _pl_hit
    if _pl_dropped:
        print(f"📱 Снят признак планшета у {_pl_dropped} обращений — слова «планшет» в тексте нет")

    # Полная очистка датафрейма для исключения "nan" в таблицах
    df_clean = df_final.fillna('')

    total_tickets = len(df_clean)
    min_cls_str = df_clean['done_dt'].min().strftime('%d.%m.%Y') if pd.notnull(df_clean['done_dt'].min()) else 'Н/Д'

    _dif_num = pd.to_numeric(df_clean['dif_Hour'], errors='coerce').dropna()
    medHour = round(float(_dif_num.median()), 1) if len(_dif_num) else 0.0

    # Топы (подготовка данных для Jinja2)
    top_initiators = df_clean.groupby('cli').agg(
        count=('id', 'count'),
        topics=('topic_cluster', lambda x: ', '.join(x.dropna().unique()[:3])),
        tickets=('id', lambda x: ', '.join(x.astype(str).tolist()))
    ).reset_index().sort_values('count', ascending=False).head(TOP_INITIATORS_N).to_dict('records')

    # ----------------------------------------------------------------------
    # ЧТО ИЗМЕНЕНО: примеры берутся из issue_summary (суть), а не из полного desc;
    #               добавлена доля инцидентов в группе
    # ПОЧЕМУ: в таблицу шла простыня текста обращения; группы, состоящие
    #         преимущественно из инцидентов, должны выделяться красным
    # ----------------------------------------------------------------------
    def _collect_samples(series):
        vals = [str(v).strip().replace('\n', ' ') for v in series.dropna() if str(v).strip()]
        vals = [v[:160] + ('…' if len(v) > 160 else '') for v in vals[:3]]
        return vals if vals else ['Нет описания']

    def _incident_share(series):
        vals = [str(v).strip().lower() for v in series]
        return round(sum(1 for v in vals if v == INCIDENT_TYPE.lower()) / len(vals) * 100) if vals else 0

    # ------------------------------------------------------------------
    # ЧТО ИЗМЕНЕНО: у группы проблем появился раздел (sec_label)
    # ПОЧЕМУ: таблица расшита до номеров обращений, а понять, чей это блок,
    #         было не по чему. Группа может задевать несколько разделов —
    #         показываем доминирующий и число остальных: «Договоры +2»
    # ------------------------------------------------------------------
    def _dominant_sec(series):
        vals = series.astype(str).str.strip().replace('', np.nan).dropna()
        if vals.empty:
            return 'Не указан'
        counts = vals.value_counts()
        rest = len(counts) - 1
        return f"{counts.index[0]} +{rest}" if rest > 0 else str(counts.index[0])

    top_topics = df_clean.groupby('topic_group').agg(
        count=('id', 'count'),
        clusters=('topic_cluster', lambda x: ', '.join(pd.Series(x).astype(str).str.strip().replace('', np.nan).dropna().unique()[:12])),
        tickets=('id', lambda x: ', '.join(x.astype(str).tolist())),
        quality=('final_grade', lambda x: x.mode()[0] if not x.mode().empty else 'Н/Д'),
        sample_descs=('issue_summary', _collect_samples),
        incident_share=('type', _incident_share),
        sec_label=('sec', _dominant_sec)
    ).reset_index().sort_values('count', ascending=False).head(TOP_PROBLEMS_N).to_dict('records')

    for _t in top_topics:
        _t['is_incident_group'] = _t['incident_share'] >= 50
    _inc_groups = sum(1 for _t in top_topics if _t['is_incident_group'])
    print(f"🏷️ Топ-{len(top_topics)} групп | инцидентных среди них: {_inc_groups}")

    longest_tickets = df_clean.sort_values('dif_Hour', ascending=False).head(TOP_LONGEST_N).to_dict('records')
    emo_tickets = df_clean[df_clean['tone_deviation_score'] > 0].sort_values('tone_deviation_score', ascending=False).to_dict('records')
    crit_tickets = df_clean[df_clean['is_critical_incident']].sort_values('reg_dt').to_dict('records')

    # ИЗМЕНЕНО: Берём 10 плохих и 10 отличных обращений
    # ----------------------------------------------------------------------
    # ЧТО ИЗМЕНЕНО (v4.2): «Плохо» — все, без .head(10)
    # ПОЧЕМУ: заголовок слайда «Плохо (N)» показывал длину урезанного списка, а не число; все плохие — на переделку (my-FT Т-47)
    # ----------------------------------------------------------------------
    bad_tickets = df_clean[df_clean['final_grade'] == 'ПЛОХО'].to_dict('records')
    exc_tickets = df_clean[df_clean['final_grade'] == 'ОТЛИЧНО'].head(10).to_dict('records')

    # Выборка обращений по планшетам для отдельного слайда
    planshet_tickets = df_clean[df_clean['planshet']].sort_values('reg_dt', ascending=False).to_dict('records')
    planshet_total = len(planshet_tickets)
    print(f"📱 Отмечено как планшеты: {planshet_total} из {total_tickets}")

    # Честный подсчет плохих тикетов (без ограничения .head(10))
    bad_count = len(df_clean[df_clean['final_grade'] == 'ПЛОХО'])

    cur_year, cur_week = int(max_year), int(max_week)

    # ----------------------------------------------------------------------
    # ЧТО ИЗМЕНЕНО: история читается из локального кэша, Диск подключается
    #               только если недели в истории ещё нет (её надо записать)
    # ПОЧЕМУ: повторный прогон той же недели не должен требовать согласия Google
    # ----------------------------------------------------------------------
    CACHE_HISTORY = os.path.join(CACHE_DIR, 'history.json')

    def _norm_week(rec):
        return {
            'year': int(rec.get('year', 0)),
            'week_num': int(rec.get('week_num', 0)),
            'total_tickets': int(rec.get('total_tickets', 0)),
            'med_hour': float(rec.get('med_hour', 0) or 0),
            'incidents': rec.get('incidents')
        }

    def load_history_cache():
        if not os.path.exists(CACHE_HISTORY):
            return {}
        try:
            with open(CACHE_HISTORY, 'r', encoding='utf-8') as f:
                return {(r['year'], r['week_num']): r for r in map(_norm_week, json.load(f))}
        except Exception as e:
            print(f"⚠️ Кэш истории повреждён, игнорирую: {e}")
            return {}

    def load_history_drive():
        if not ensure_drive('нужна история недель'):
            return {}
        out = {}
        for hf in glob.glob(os.path.join(HISTORY_DIR, "week_*.json")):
            try:
                with open(hf, 'r', encoding='utf-8') as f:
                    rec = _norm_week(json.load(f))
                out[(rec['year'], rec['week_num'])] = rec
            except Exception as e:
                print(f"⚠️ Пропущен повреждённый файл истории {os.path.basename(hf)}: {e}")
        return out

    def save_history(hist_map):
        """Кэш — всегда. Диск — только если запись новая или цифры изменились.

        ЧТО ИЗМЕНЕНО: добавлено сравнение с уже сохранённой неделей
        ПОЧЕМУ: повторный прогон той же выгрузки не должен дёргать согласие Google
        """
        rec = hist_map[(cur_year, cur_week)]
        prev = _SAVED_WEEK.get('rec')
        with open(CACHE_HISTORY, 'w', encoding='utf-8') as f:
            json.dump([hist_map[k] for k in sorted(hist_map)], f, ensure_ascii=False, indent=2)

        if prev is not None and all(prev.get(k) == rec.get(k) for k in ('total_tickets', 'med_hour', 'incidents')):
            print("♻️ Цифры недели не изменились — Диск не трогаем")
            return _DRIVE_READY['ok']

        if ensure_drive('запись истории недели'):
            with open(os.path.join(HISTORY_DIR, f"week_{cur_year}_W{cur_week:02d}.json"), 'w', encoding='utf-8') as f:
                json.dump(rec, f, ensure_ascii=False, indent=2)
            return True
        return False

    hist_map = load_history_cache()
    # Запоминаем, что уже было сохранено за эту неделю — для сравнения при записи
    _SAVED_WEEK = {'rec': hist_map.get((cur_year, cur_week))}
    week_is_new = (cur_year, cur_week) not in hist_map

    if week_is_new and DRIVE_MODE != "never":
        # Недели нет в кэше — возможно, она есть на Диске с прошлой сессии
        drive_map = load_history_drive()
        if drive_map:
            hist_map.update(drive_map)
            week_is_new = (cur_year, cur_week) not in hist_map
    if not week_is_new:
        print(f"♻️ Неделя {cur_year}-W{cur_week:02d} уже в истории — Диск не требуется")

    past_weeks = [hist_map[k] for k in sorted(hist_map) if k < (cur_year, cur_week)]
    prev_week_data = past_weeks[-1] if past_weeks else None
    w1_t = prev_week_data['total_tickets'] if prev_week_data else None
    w1_m = prev_week_data['med_hour'] if prev_week_data else None

    current_week_data = {"year": cur_year, "week_num": cur_week,
                         "total_tickets": int(total_tickets), "med_hour": float(medHour),
                         "incidents": None}
    hist_map[(cur_year, cur_week)] = current_week_data
    hist_data = [hist_map[k] for k in sorted(hist_map)]

    _prev_label = f"{prev_week_data['year']}-W{prev_week_data['week_num']:02d}" if prev_week_data else 'нет'
    print(f"📅 История: недель в графике {len(hist_data)} | W-1 = {_prev_label}")

    # Контекст для аналитической модели (Root Cause)
    bad_summary_text = "\n".join([f"- ID {r['id']} ({r['topic_cluster']}): {r['desc']} | Ответ: {r['sol']} | Ошибка: {r['grade_argument']}" for r in bad_tickets[:5]])
    crit_summary_text = "\n".join([f"- ID {r['id']}: {r['topic_cluster']} ({r['desc']})" for r in crit_tickets[:5]])
    top_topics_text = ", ".join([f"{t['topic_group']} ({t['count']} шт)" for t in top_topics[:5]])

    summary_for_ai = f"""СТАТИСТИКА НЕДЕЛИ:
    - Всего обращений: {total_tickets}
    - Медианное время решения: {medHour} ч.
    - Топ проблемных групп: {top_topics_text}
    - Критических инцидентов: {len(crit_tickets)}
    - Обращений с оценкой ПЛОХО: {bad_count}
    КЛЮЧЕВЫЕ КРИТИЧЕСКИЕ СБОИ:
    {crit_summary_text if crit_summary_text else 'Нет'}
    ПРИМЕРЫ ПЛОХИХ РЕШЕНИЙ АНАЛИТИКОВ:
    {bad_summary_text if bad_summary_text else 'Нет'}"""

    summary_tokens = {'prompt': 0, 'candidates': 0, 'total': 0}
    summary_prompt = f"Ты — главный архитектор систем 1С. Дай 3 глубоких вывода об архитектуре процесса и Root Cause: {summary_for_ai}. Формат HTML <li>"

    try:
        print(f"🧠 Генерация выводов Архитектора ({AI_PROVIDER.upper()} -> {SUMMARY_MODEL})...")
        if AI_PROVIDER == "google":
            response = client_gemini.models.generate_content(
                model=SUMMARY_MODEL,
                contents=summary_prompt,
                config={'temperature': 0.2}
            )
            if response.usage_metadata:
                summary_tokens['prompt'] = response.usage_metadata.prompt_token_count
                summary_tokens['candidates'] = response.usage_metadata.candidates_token_count
                summary_tokens['total'] = response.usage_metadata.total_token_count
            ai_conclusions = response.text.replace('```html', '').replace('```', '').strip()
        else: # deepseek
            response = client_deepseek_sync.chat.completions.create(
                model=SUMMARY_MODEL,
                messages=[
                    {"role": "system", "content": "Ты — ведущий аналитик и архитектор 1С. Отвечай только набором HTML-тегов <li> без внешних блоков markdown."},
                    {"role": "user", "content": summary_prompt}
                ],
                temperature=0.2
            )
            usage = response.usage
            if usage:
                summary_tokens['prompt'] = usage.prompt_tokens
                summary_tokens['candidates'] = usage.completion_tokens
                summary_tokens['total'] = usage.total_tokens
            ai_conclusions = response.choices[0].message.content.replace('```html', '').replace('```', '').strip()

        print(f"✅ Выводы сгенерированы. Потреблено токенов ({SUMMARY_MODEL}): {summary_tokens['total']}")
    except Exception as e:
        # ----------------------------------------------------------------------
        # ЧТО ИЗМЕНЕНО (v4.2): при сбое ИИ в отчёт идёт нейтральная строка, текст ошибки — только в журнал запуска
        # ПОЧЕМУ: текст исключения попадал в слайд: читателю отчёта он не нужен и может раскрывать детали API
        # ----------------------------------------------------------------------
        print(f"⚠️ Выводы ИИ не сгенерированы ({AI_PROVIDER}): {e}")
        ai_conclusions = "<li>Выводы ИИ недоступны: сбой модели при генерации. Подробности — в журнале запуска.</li>"
else:
    print("⏭️ ПРОПУСК: Отчет (ENABLE_MAIN = False).")

# БЛОК 5А: СТАТУСЫ, ТИПЫ ОБРАЩЕНИЙ, ИНЦИДЕНТЫ

In [ ]:
# =====================================================================
# [ЯЧЕЙКА 5А] РАЗРЕЗЫ ДЛЯ СЛАЙДОВ: ТАЙМЛАЙНЫ, ОТКРЫТЫЕ, ТИПЫ, ИНЦИДЕНТЫ
# =====================================================================
# ЧТО ИЗМЕНЕНО: ячейка переписана
# ПОЧЕМУ: добавились таймлайн медиан, плитка типов, статистика, расширенные
#         инициаторы, неверная классификация, возрастные корзины и валидация
# =====================================================================
if ENABLE_MAIN:

    # ------------------------------------------------------------------
    # 0. Вспомогательные функции
    # ------------------------------------------------------------------
    def build_matrix(frame, row_col, col_col):
        """Матрица счётчиков: строки — row_col, колонки — col_col."""
        if frame.empty:
            return {'cols': [], 'rows': [], 'totals': [], 'total': 0}
        rows_src = frame[row_col].astype(str).str.strip().replace('', 'Без блока')
        cols_src = frame[col_col].astype(str).str.strip().replace('', 'Не указан')
        ct = pd.crosstab(rows_src, cols_src)
        ct = ct.loc[ct.sum(axis=1).sort_values(ascending=False).index]
        ct = ct[ct.sum(axis=0).sort_values(ascending=False).index]
        return {
            'cols': [str(c) for c in ct.columns],
            'rows': [{'name': str(i), 'cells': [int(v) for v in ct.loc[i]], 'total': int(ct.loc[i].sum())} for i in ct.index],
            'totals': [int(ct[c].sum()) for c in ct.columns],
            'total': int(ct.values.sum())
        }

    def type_color(type_name):
        return TYPE_COLORS.get(str(type_name).strip(), TYPE_COLOR_DEFAULT)

    def hours_between(a, b):
        """Рабочие часы между двумя датами (календарные / 3)."""
        d = (b - a).dt.total_seconds() / 3600.0 / SLA_DIVIDER
        return d[d.notna() & (d >= 0)]

    # ------------------------------------------------------------------
    # 1. Плитка «Всего» — состав по типам обращений
    # ------------------------------------------------------------------
    _type_counts = df_all['type'].astype(str).str.strip().replace('', 'Не указан').value_counts()
    total_all = int(_type_counts.sum())
    type_tiles = [{
        'name': str(k),
        'count': int(v),
        'pct': round(v / total_all * 100, 1) if total_all else 0,
        'color': type_color(k)
    } for k, v in _type_counts.items()]
    _tiles_log = ', '.join(['{0} — {1}'.format(t['name'], t['count']) for t in type_tiles])
    print('🎨 Плитка типов: ' + _tiles_log)

    # ------------------------------------------------------------------
    # 1.1 Суточные ряды отчётной недели: решений предложено и зарегистрировано
    # ЧТО ИЗМЕНЕНО: новый блок
    # ПОЧЕМУ: на первом слайде нужны два столбчатых графика по дням пн–вс;
    #         дни без данных обязаны присутствовать нулями, иначе график «съедет»
    # ------------------------------------------------------------------
    _WD_SHORT = ['Пн', 'Вт', 'Ср', 'Чт', 'Пт', 'Сб', 'Вс']

    def daily_series(dates, label):
        """Счётчики по дням отчётной недели. Возвращает 7 точек, включая пустые дни."""
        d = pd.to_datetime(dates, errors='coerce').dropna()
        d = d[(d >= WEEK_START) & (d < WEEK_END_EXCL)]
        by_day = d.dt.normalize().value_counts()
        points = [{
            'day': _WD_SHORT[i],
            'date': day.strftime('%d.%m'),
            'count': int(by_day.get(day, 0)),
            'is_weekend': i >= 5
        } for i, day in enumerate(WEEK_DAYS)]
        peak = max((p['count'] for p in points), default=0)
        for p in points:
            p['pct'] = round(p['count'] / peak * 100, 1) if peak else 0
        return {'label': label, 'points': points, 'total': sum(p['count'] for p in points), 'peak': peak}

    daily_closed = daily_series(df_closed['done_dt'], 'Предложено решений')
    daily_reg = daily_series(df_all['reg_dt'], 'Зарегистрировано')
    print(f"📅 По дням недели: решений {daily_closed['total']}, регистраций {daily_reg['total']}")

    # ------------------------------------------------------------------
    # 2. Таймлайны медианы и среднего: reg → res → cls
    # ------------------------------------------------------------------
    _to_res = hours_between(df_clean['reg_dt'], df_clean['res_dt'])
    _res_to_cls = hours_between(df_clean['res_dt'], df_clean['cls_dt'])
    _to_cls = hours_between(df_clean['reg_dt'], df_clean['cls_dt'])

    def make_timeline(fn, label):
        return {
            'label': label,
            'to_res': round(float(fn(_to_res)), 1) if len(_to_res) else 0.0,
            'res_to_cls': round(float(fn(_res_to_cls)), 1) if len(_res_to_cls) else 0.0,
            'to_cls': round(float(fn(_to_cls)), 1) if len(_to_cls) else 0.0,
            'n_res': int(len(_to_res)),
            'n_cls': int(len(_to_cls))
        }

    timeline_median = make_timeline(lambda s: s.median(), 'Медиана')
    timeline_mean = make_timeline(lambda s: s.mean(), 'Среднее')
    print(f"⏱️ Медиана: рег→решение {timeline_median['to_res']} ч | решение→закрытие {timeline_median['res_to_cls']} ч "
          f"(подтверждено {timeline_median['n_cls']} из {len(df_clean)})")

    # ------------------------------------------------------------------
    # 3. Распределение по группам тем — горизонтальные столбцы
    # ------------------------------------------------------------------
    CHART_TOP_N = 10
    _g = df_clean.groupby('topic_group').agg(
        count=('id', 'count'),
        inc=('type', lambda x: sum(1 for v in x if str(v).strip().lower() == INCIDENT_TYPE.lower()))
    ).sort_values('count', ascending=False)
    _max_cnt = int(_g['count'].max()) if len(_g) else 1
    group_bars = [{
        'name': str(i)[:60],
        'count': int(r['count']),
        'inc': int(r['inc']),
        'pct': round(r['count'] / len(df_clean) * 100, 1) if len(df_clean) else 0,
        'width': round(r['count'] / _max_cnt * 100),
        'inc_width': round(r['inc'] / _max_cnt * 100)
    } for i, r in _g.head(CHART_TOP_N).iterrows()]
    group_rest = int(_g['count'].iloc[CHART_TOP_N:].sum()) if len(_g) > CHART_TOP_N else 0

    # ------------------------------------------------------------------
    # 4. Общая статистика недели
    # ------------------------------------------------------------------
    _dif_all = pd.to_numeric(df_clean['dif_Hour'], errors='coerce').dropna()
    # ----------------------------------------------------------------------
    # ЧТО ИЗМЕНЕНО (v4.2): «в среднем за день» — решённые за неделю / рабочие дни недели до последней даты решения; «пик» — по дню решения
    # ПОЧЕМУ: раньше знаменатель — число разных дат регистрации решённых (они могут быть месяцами раньше), пик — по дню регистрации
    # ----------------------------------------------------------------------
    _days = [d for d in WEEK_DAYS if d.weekday() < 5 and d <= _last_done.normalize()]
    _weekday_names = ['Понедельник', 'Вторник', 'Среда', 'Четверг', 'Пятница', 'Суббота', 'Воскресенье']
    _by_weekday = df_clean['done_dt'].dt.weekday.value_counts()
    _repeat_groups = _g[_g['count'] > 1]
    _open_ages = (REPORT_DATE - df_open['reg_dt']).dt.days

    _longest = df_clean.sort_values('dif_Hour', ascending=False).iloc[0] if len(df_clean) else None

    week_stats = [
        {'icon': '⚡', 'label': 'Решено за рабочий день',
         'value': f"{round((_dif_all <= FAST_SOLVE_HOURS).mean() * 100)}%",
         'note': f"{int((_dif_all <= FAST_SOLVE_HOURS).sum())} из {len(_dif_all)} обращений"},
        {'icon': '🐌', 'label': 'Долгострой (> 40 ч)',
         'value': f"{int((_dif_all > LONG_SOLVE_HOURS).sum())}",
         'note': (f"дольше всех — {_longest['id']} ({_longest['dif_Hour']} ч)" if _longest is not None else '—')},
        {'icon': '📅', 'label': 'В среднем за день',
         'value': f"{round(len(df_clean) / max(len(_days), 1), 1)}",
         'note': (f"пик — {_weekday_names[int(_by_weekday.index[0])]} ({int(_by_weekday.iloc[0])} шт.)" if len(_by_weekday) else '—')},
        {'icon': '🔁', 'label': 'Повторяющиеся проблемы',
         'value': f"{round(int(_repeat_groups['count'].sum()) / len(df_clean) * 100) if len(df_clean) else 0}%",
         'note': f"{len(_repeat_groups)} групп с повтором из {len(_g)}"},
        {'icon': '📂', 'label': 'Открытых сейчас',
         'value': f"{len(df_open)}",
         'note': f"старше месяца — {int((_open_ages > OPEN_AGE_DAYS).sum())}"},
        {'icon': '🚨', 'label': 'Инцидентов в работе',
         'value': f"{int((df_open['type'].str.lower() == INCIDENT_TYPE.lower()).sum())}",
         'note': f"из {int((df_all['type'].str.lower() == INCIDENT_TYPE.lower()).sum())} всего"},
    ]

    # ------------------------------------------------------------------
    # 5. Топ-5 инициаторов с разбором по сути
    # ------------------------------------------------------------------
    top_initiators_ext = []
    for cli, grp in df_clean.groupby('cli'):
        by_sense = grp.groupby('issue_summary').agg(
            cnt=('id', 'count'),
            ids=('id', lambda x: ', '.join(x.astype(str).tolist())),
            sec=('sec', lambda x: x.mode()[0] if not x.mode().empty else '')
        ).reset_index().sort_values('cnt', ascending=False)
        top_initiators_ext.append({
            'cli': str(cli) if str(cli).strip() else 'Не указан',
            'count': len(grp),
            'blocks': ', '.join(grp['sec'].astype(str).str.strip().replace('', np.nan).dropna().unique()[:4]),
            'incidents': int((grp['type'].str.lower() == INCIDENT_TYPE.lower()).sum()),
            'senses': [{'sense': str(r['issue_summary'])[:120], 'cnt': int(r['cnt']),
                        'ids': r['ids'], 'sec': str(r['sec'])} for _, r in by_sense.head(8).iterrows()]
        })
    top_initiators_ext = sorted(top_initiators_ext, key=lambda x: -x['count'])[:TOP_INITIATORS_N]

    # ------------------------------------------------------------------
    # 6. Неверно классифицированные обращения
    # ------------------------------------------------------------------
    _mis = df_clean[
        (df_clean['suggested_type'] != '') &
        (df_clean['suggested_type'].str.lower() != df_clean['type'].str.lower())
    ].copy()
    misclassified_rows = [{
        'id': str(r['id']),
        'reg': r['reg_dt'].strftime('%d.%m.%Y') if pd.notnull(r['reg_dt']) else 'Н/Д',
        'type_old': str(r['type']),
        'type_new': str(r['suggested_type']),
        'color_old': type_color(r['type']),
        'color_new': type_color(r['suggested_type']),
        'sec': str(r['sec']),
        'summary': str(r['issue_summary'])[:200],
        'argument': str(r['type_argument'])[:400] or 'Тип определён по существу текста обращения'
    } for _, r in _mis.sort_values('reg_dt').iterrows()]
    misclassified_total = len(misclassified_rows)
    print(f"🔀 Неверно классифицировано: {misclassified_total} из {len(df_clean)}")

    # ------------------------------------------------------------------
    # 7. Открытые обращения: на чистку и по возрасту
    # ------------------------------------------------------------------
    _open = df_open.copy()
    _open['age_days'] = (REPORT_DATE - _open['reg_dt']).dt.days

    open_old = _open[_open['age_days'] > OPEN_AGE_DAYS]
    open_new = _open[_open['age_days'] <= OPEN_AGE_DAYS]
    open_total = len(_open)

    # 7.1 Таблица "на чистку": старше месяца и НЕ в рабочих статусах
    _cleanup = open_old[~open_old['st'].isin(WORKFLOW_STATUSES)].sort_values('reg_dt')
    cleanup_rows = [{
        'id': str(r['id']),
        'reg': r['reg_dt'].strftime('%d.%m.%Y') if pd.notnull(r['reg_dt']) else 'Н/Д',
        'age': int(r['age_days']) if pd.notnull(r['age_days']) else 0,
        'st': str(r['st']),
        'sec': str(r['sec']),
        'desc': str(r.get('desc', '') or '')[:300]
    } for _, r in _cleanup.iterrows()]
    cleanup_total = len(cleanup_rows)

    # 7.2 Возрастные корзины — единая модель на весь отчёт
    # ЧТО ИЗМЕНЕНО: было 5 корзин и таблица «статус × корзина», стало 4 корзины
    #               и столбчатый график
    # ПОЧЕМУ: корзины «1–2 месяца» и «более 2 месяцев» читались как одно и то же —
    #         «висит давно»; та же модель теперь применяется и к инцидентам
    AGE_BUCKETS4 = [
        ('Текущая неделя', lambda d: d >= WEEK_START),
        ('1–2 недели', lambda d: (d < WEEK_START) & (d >= WEEK_START - pd.Timedelta(days=7))),
        ('2 недели – месяц', lambda d: (d < WEEK_START - pd.Timedelta(days=7)) & (d >= REPORT_DATE - pd.Timedelta(days=30))),
        ('Более месяца', lambda d: d < REPORT_DATE - pd.Timedelta(days=30)),
    ]
    # Палитра корзин проверена валидатором: контраст к фону >=3:1, различима при дальтонизме
    AGE_BUCKET_COLORS = ['#0f6e3a', '#1a56db', '#c77700', '#c5221f']

    def age_bars(frame, label):
        """Распределение по 4 возрастным корзинам от даты регистрации."""
        cells = [int(cond(frame['reg_dt']).sum()) for _, cond in AGE_BUCKETS4]
        peak = max(cells) if cells else 0
        total = sum(cells)
        return {
            'label': label,
            'total': total,
            'bars': [{
                'name': AGE_BUCKETS4[i][0],
                'count': cells[i],
                'pct_of_total': round(cells[i] / total * 100, 1) if total else 0,
                'pct': round(cells[i] / peak * 100, 1) if peak else 0,
                'color': AGE_BUCKET_COLORS[i]
            } for i in range(len(cells))]
        }

    age_bars4 = age_bars(_open, 'Открытые обращения')
    age_cols = [name for name, _ in AGE_BUCKETS4]
    print(f"📂 Открытых {open_total} | на чистку {cleanup_total} | по корзинам " +
          ', '.join(f"{b['name']}: {b['count']}" for b in age_bars4['bars']))

    open_matrix_new = build_matrix(open_new, 'sec', 'st')

    # ------------------------------------------------------------------
    # 8. Типы обращений — только закрытые
    # ------------------------------------------------------------------
    type_matrix = build_matrix(df_clean, 'sec', 'type')

    # ------------------------------------------------------------------
    # 9. Инциденты: закрытые и открытые по отдельности + таймлайн
    # ------------------------------------------------------------------
    _crit_ids = set(df_clean[df_clean['is_critical_incident']]['id'].astype(str))
    _sum_map = df_clean.set_index('id')['issue_summary'].to_dict()
    _grade_map = df_clean.set_index('id')['final_grade'].to_dict()

    def incident_row(r):
        tid = str(r['id'])
        return {
            'id': tid,
            'reg': r['reg_dt'].strftime('%d.%m.%Y') if pd.notnull(r['reg_dt']) else 'Н/Д',
            'reg_short': r['reg_dt'].strftime('%d.%m') if pd.notnull(r['reg_dt']) else '',
            'st': str(r['st']),
            'sec': str(r['sec']),
            'summary': _sum_map.get(tid, str(r.get('desc', ''))[:120]),
            'desc': str(r.get('desc', '') or '')[:1200],
            'sol': str(r.get('sol', '') or '')[:1200],
            'cli': str(r.get('cli', '') or ''),
            'anl': str(r.get('anl', '') or ''),
            'grade': _grade_map.get(tid, ''),
            'is_closed': str(r['st']) in CLOSED_STATUSES,
            'is_crit': tid in _crit_ids
        }

    _inc_all = df_all[df_all['type'].str.lower() == INCIDENT_TYPE.lower()].copy()
    _inc_closed = _inc_all[_inc_all['st'].isin(CLOSED_STATUSES)].sort_values(['st', 'reg_dt'])
    _inc_open = _inc_all[~_inc_all['st'].isin(CLOSED_STATUSES)].sort_values(['st', 'reg_dt'])

    incidents_closed = [incident_row(r) for _, r in _inc_closed.iterrows()]
    incidents_open = [incident_row(r) for _, r in _inc_open.iterrows()]
    incident_rows = incidents_closed + incidents_open
    incident_total = len(incident_rows)
    incident_crit = sum(1 for r in incident_rows if r['is_crit'])
    incident_open_cnt = len(incidents_open)

    # 9.1 Таблица критичных инцидентов — всё, что помечено как критичное
    # ЧТО ИЗМЕНЕНО: новый блок
    # ПОЧЕМУ: критичные тонули среди 67 строк общих таблиц; фильтр недели к ним
    #         намеренно не применяется — критичное не должно исчезать со слайда
    crit_incident_rows = [r for r in incident_rows if r['is_crit']]
    print(f"🔴 Критичных инцидентов в таблице: {len(crit_incident_rows)}")

    # 9.2 Возрастные корзины по открытым инцидентам (та же модель, что у открытых)
    inc_age_bars = age_bars(_inc_open, 'Открытые инциденты')

    # 9.3 Точки для SVG-таймлайна — только инциденты, зарегистрированные на неделе
    # ЧТО ИЗМЕНЕНО: добавлен фильтр по отчётной неделе
    # ПОЧЕМУ: раньше на ось ложились все 67 инцидентов с 2022 года, и точки
    #         слипались в неразличимую кашу у левого края
    _tl = _inc_all.dropna(subset=['reg_dt'])
    _tl = _tl[(_tl['reg_dt'] >= WEEK_START) & (_tl['reg_dt'] < WEEK_END_EXCL)].sort_values('reg_dt')
    timeline_points = []
    if len(_tl):
        _t0, _t1 = _tl['reg_dt'].min(), _tl['reg_dt'].max()
        _span = max((_t1 - _t0).total_seconds(), 1)
        for pos, (_, r) in enumerate(_tl.iterrows()):
            tid = str(r['id'])
            timeline_points.append({
                'id': tid,
                'x': round(40 + (r['reg_dt'] - _t0).total_seconds() / _span * 920),
                'up': pos % 2 == 0,
                'date': r['reg_dt'].strftime('%d.%m'),
                'is_crit': tid in _crit_ids,
                'is_closed': str(r['st']) in CLOSED_STATUSES,
                'sec': str(r['sec'])
            })
    timeline_from = _tl['reg_dt'].min().strftime('%d.%m.%Y') if len(_tl) else '—'
    timeline_to = _tl['reg_dt'].max().strftime('%d.%m.%Y') if len(_tl) else '—'
    print(f"🚨 Инцидентов {incident_total}: закрытых {len(incidents_closed)}, открытых {incident_open_cnt}, критичных {incident_crit}")

    # ------------------------------------------------------------------
    # 9.4 История недель: дописываем число инцидентов
    # ЧТО ИЗМЕНЕНО: в файл недели добавлено поле incidents
    # ПОЧЕМУ: на графике динамики нужна вторая линия; за прошлые недели
    #         данных нет — начинаем вести с текущей
    # ------------------------------------------------------------------
    if TEST_SAMPLE_SIZE == 0:
        # ----------------------------------------------------------------------
        # ЧТО ИЗМЕНЕНО (v4.2): в историю пишутся инциденты, решённые за неделю, и метка hist_ver = 2
        # ПОЧЕМУ: раньше писалось число инцидентов всей выгрузки — красная линия динамики показывала накопленный остаток; по метке v5 отличит верные записи от старых
        # ----------------------------------------------------------------------
        current_week_data['incidents'] = int((df_clean['type'].str.lower() == INCIDENT_TYPE.lower()).sum())
        current_week_data['hist_ver'] = 2
        hist_map[(cur_year, cur_week)] = current_week_data
        _saved = save_history(hist_map)
        print(f"💾 История недели сохранена: кэш{' и Google Диск' if _saved else ' (Диск недоступен — скачайте history.json в конце)'}")
    hist_data = [hist_map[k] for k in sorted(hist_map)]

    # ------------------------------------------------------------------
    # 10. Планшеты: открытые и закрытые
    # ------------------------------------------------------------------
    # ЧТО ИЗМЕНЕНО: планшеты ищутся по тексту во ВСЕЙ выгрузке, а не только
    #               среди закрытых обращений, размеченных ИИ
    # ПОЧЕМУ: признак чисто текстовый (слово «планшет»), а ИИ размечает только
    #         закрытые — открытые обращения по планшетам в слайд не попадали
    _pl_text_all = (df_all['sec'].astype(str) + ' ' + df_all['desc'].astype(str) + ' ' + df_all['sol'].astype(str)).str.lower()
    _pl = df_all[_pl_text_all.str.contains('планшет', na=False)].copy()
    planshet_rows = [{
        'id': str(r['id']),
        'reg': r['reg_dt'].strftime('%d.%m.%Y') if pd.notnull(r['reg_dt']) else 'Н/Д',
        'st': str(r['st']),
        'sec': str(r['sec']),
        'summary': _sum_map.get(str(r['id']), ''),
        'desc': str(r.get('desc', '') or '')[:800],
        'sol': str(r.get('sol', '') or '')[:800],
        'cli': str(r.get('cli', '') or ''),
        'grade': _grade_map.get(str(r['id']), ''),
        'is_closed': str(r['st']) in CLOSED_STATUSES
    } for _, r in _pl.sort_values('reg_dt', ascending=False).iterrows()]
    planshet_total = len(planshet_rows)
    planshet_closed = sum(1 for r in planshet_rows if r['is_closed'])
    print(f"📱 Обращений со словом «планшет»: {planshet_total} (закрытых {planshet_closed}, остальных {planshet_total - planshet_closed})")

    # ------------------------------------------------------------------
    # 11. Валидация: сходимость чисел между слайдами
    # ------------------------------------------------------------------
    # ЧТО ИЗМЕНЕНО: проверки пересчитаны под рез по отчётной неделе
    # ПОЧЕМУ: после фильтра «строк в выгрузке = закрытые + открытые + отклонённые»
    #         перестало сходиться — часть строк лежит за границами недели
    _closed_by_status = int(df_all['st'].isin(CLOSED_STATUSES).sum())
    _rejected_all = int((df_all['st'] == REJECTED_STATUS).sum())
    _out_rejected = _rejected_all - len(df_rejected)
    _rej = df_all[df_all['st'] == REJECTED_STATUS]
    _rej_in = int(_in_week(_rej['cls_dt']).sum())
    _rej_out = int((_rej['cls_dt'].notna() & ~_in_week(_rej['cls_dt'])).sum())
    _rej_nodate = int(_rej['cls_dt'].isna().sum())
    _graded = int((~df_clean['final_grade'].isin(['', 'Н/Д'])).sum())
    validation_rows = [
        {'name': 'Строк в выгрузке = завершённые за неделю + вне недели + открытые',
         'left': len(df_all),
         'right': len(df_clean) + len(df_rejected) + OUT_OF_WEEK_CLOSED + _out_rejected + len(df_open),
         'ok': len(df_all) == len(df_clean) + len(df_rejected) + OUT_OF_WEEK_CLOSED + _out_rejected + len(df_open)},
        {'name': 'Закрытых за неделю + вне недели = строк со статусами «Закрыто» / «Отработано»',
         'left': len(df_clean) + OUT_OF_WEEK_CLOSED, 'right': _closed_by_status,
         'ok': len(df_clean) + OUT_OF_WEEK_CLOSED == _closed_by_status},
        # ----------------------------------------------------------------------
        # ЧТО ИЗМЕНЕНО (v4.2): проверки №3 и №4 считаются по факту
        # ПОЧЕМУ: №3 сходилась по построению (правая часть выводилась из левой), №4 после v4.1 засчитывала «Н/Д» как оценку — обе всегда были зелёными
        # ----------------------------------------------------------------------
        {'name': 'Отклонённых всего = в неделе + вне недели с датой + без даты (пересчёт по выгрузке)',
         'left': _rejected_all,
         'right': _rej_in + _rej_out + _rej_nodate,
         'ok': _rejected_all == _rej_in + _rej_out + _rej_nodate and _rej_in == len(df_rejected)},
        {'name': 'Оценки ИИ проставлены всем закрытым (без «Н/Д»)',
         'left': _graded, 'right': len(df_clean),
         'ok': _graded == len(df_clean)},
        {'name': 'Инциденты = закрытые + открытые',
         'left': incident_total, 'right': len(incidents_closed) + incident_open_cnt,
         'ok': incident_total == len(incidents_closed) + incident_open_cnt},
        {'name': 'Открытые = старше месяца + младше месяца',
         'left': open_total, 'right': len(open_old) + len(open_new),
         'ok': open_total == len(open_old) + len(open_new)},
        {'name': 'Сумма по типам = всего обращений',
         'left': sum(t['count'] for t in type_tiles), 'right': len(df_all),
         'ok': sum(t['count'] for t in type_tiles) == len(df_all)},
    ]
    validation_ok = all(r['ok'] for r in validation_rows)
    print(f"{'✅' if validation_ok else '❌'} Валидация: {sum(1 for r in validation_rows if r['ok'])} из {len(validation_rows)} проверок пройдено")
else:
    print("⏭️ ПРОПУСК: Отчет (ENABLE_MAIN = False).")

# БЛОК 5Б: АНАЛИЗ ТАБЛИЦЫ ИНЦИДЕНТОВ

In [ ]:
# =====================================================================
# [ЯЧЕЙКА 5Б] КОММЕНТАРИИ ИИ К СЛАЙДАМ
# =====================================================================
# ЧТО ИЗМЕНЕНО: ячейка переписана — было 1 обращение к модели, стало 5
# ПОЧЕМУ: нужны разборы по инцидентам (закрытые / открытые / общий),
#         комментарий к инициаторам и к таблице типов обращений
# =====================================================================
ai_comments = {}
incidents_tokens = {'prompt': 0, 'candidates': 0, 'total': 0}

if ENABLE_MAIN:

    def ask_analyst(prompt, system="Ты — ведущий аналитик и архитектор 1С. "
                                  "Отвечай только HTML-тегами <p>, <ul>, <li>, <b>, без markdown и без внешних блоков."):
        """Один вызов аналитической модели. Возвращает HTML-фрагмент."""
        global incidents_tokens
        try:
            if AI_PROVIDER == "google":
                resp = client_gemini.models.generate_content(
                    model=SUMMARY_MODEL, contents=f"{system}\n\n{prompt}", config={'temperature': 0.2})
                if resp.usage_metadata:
                    incidents_tokens['prompt'] += resp.usage_metadata.prompt_token_count
                    incidents_tokens['candidates'] += resp.usage_metadata.candidates_token_count
                    incidents_tokens['total'] += resp.usage_metadata.total_token_count
                text = resp.text
            else:
                resp = client_deepseek_sync.chat.completions.create(
                    model=SUMMARY_MODEL,
                    messages=[{"role": "system", "content": system},
                              {"role": "user", "content": prompt}],
                    temperature=0.2
                )
                if resp.usage:
                    incidents_tokens['prompt'] += resp.usage.prompt_tokens
                    incidents_tokens['candidates'] += resp.usage.completion_tokens
                    incidents_tokens['total'] += resp.usage.total_tokens
                text = resp.choices[0].message.content
            return text.replace('```html', '').replace('```', '').strip()
        except Exception as e:
            # ----------------------------------------------------------------------
            # ЧТО ИЗМЕНЕНО (v4.2): при сбое ИИ — нейтральная строка в отчёт, ошибка — в журнал
            # ПОЧЕМУ: текст исключения попадал прямо в слайд
            # ----------------------------------------------------------------------
            print(f"⚠️ Комментарий ИИ не сгенерирован ({AI_PROVIDER}): {e}")
            return "<p>Разбор ИИ недоступен: сбой модели при генерации. Подробности — в журнале запуска.</p>"

    def incidents_digest(rows, limit=45):
        return "\n".join([
            f"- {r['id']} | {r['reg']} | {r['st']} | блок: {r['sec']} | аналитик: {r['anl']}\n"
            f"  Суть: {r['summary']}\n  Решение: {r['sol'][:300]}"
            for r in rows[:limit]
        ]) or 'Нет данных'

    # --- 1. Инциденты: закрытые ---
    if incidents_closed:
        _blocks = pd.Series([r['sec'] for r in incidents_closed]).value_counts()
        _top_block = f"{_blocks.index[0]} ({int(_blocks.iloc[0])} шт.)" if len(_blocks) else '—'
        print(f"🧠 [1/5] Разбор закрытых инцидентов ({len(incidents_closed)})...")
        ai_comments['inc_closed'] = ask_analyst(
            f"Ниже закрытые инциденты за период ({len(incidents_closed)} шт.). "
            f"Больше всего их в блоке {_top_block}.\n\n{incidents_digest(incidents_closed)}\n\n"
            "Разбери развёрнуто: какие инциденты повторяются или имеют общую техническую первопричину; "
            "что говорит концентрация в блоке-лидере — это дефект конкретного объекта системы или процесса; "
            "какие из решений были разовыми заплатками, которые вернут эту же проблему; "
            "какие доработки закроют сразу группу инцидентов. Пиши конкретно, с номерами обращений."
        )
    else:
        ai_comments['inc_closed'] = "<p>Закрытых инцидентов за период нет.</p>"

    # --- 2. Инциденты: открытые ---
    if incidents_open:
        _st = pd.Series([r['st'] for r in incidents_open]).value_counts()
        _st_txt = ', '.join([f"{k} — {int(v)}" for k, v in _st.items()])
        print(f"🧠 [2/5] Разбор открытых инцидентов ({len(incidents_open)})...")
        ai_comments['inc_open'] = ask_analyst(
            f"Ниже НЕЗАКРЫТЫЕ инциденты ({len(incidents_open)} шт.). Распределение по статусам: {_st_txt}.\n\n"
            f"{incidents_digest(incidents_open)}\n\n"
            "Разбери развёрнуто: что из этого требует эскалации в первую очередь и почему; "
            "где видно, что обращение застряло не по технической причине, а из-за процесса; "
            "какие открытые инциденты связаны между собой и должны решаться одной доработкой; "
            "что грозит бизнесу, если оставить как есть. Пиши конкретно, с номерами обращений."
        )
    else:
        ai_comments['inc_open'] = "<p>Незакрытых инцидентов нет.</p>"

    # --- 3. Инциденты: общий вывод ---
    if incident_rows:
        print("🧠 [3/5] Общий вывод по инцидентам...")
        _blocks_all = pd.Series([r['sec'] for r in incident_rows]).value_counts().head(5)
        _blocks_txt = ', '.join([f"{k} — {int(v)}" for k, v in _blocks_all.items()])
        ai_comments['inc_total'] = ask_analyst(
            f"Итог по инцидентам за период: всего {incident_total}, закрыто {len(incidents_closed)}, "
            f"открыто {incident_open_cnt}, отмечено критичными {incident_crit}. "
            f"Распределение по функциональным блокам: {_blocks_txt}.\n\n"
            f"Закрытые:\n{incidents_digest(incidents_closed, 25)}\n\nОткрытые:\n{incidents_digest(incidents_open, 25)}\n\n"
            "Дай общий вывод: есть ли системная закономерность в том, где возникают инциденты; "
            "какой блок системы требует архитектурного вмешательства, а не точечных исправлений; "
            "как соотносятся скорость закрытия и повторяемость; три приоритетных шага на следующую неделю."
        )
    else:
        ai_comments['inc_total'] = "<p>Инцидентов за период не зарегистрировано.</p>"

    # --- 4. Инициаторы ---
    if top_initiators_ext:
        print("🧠 [4/5] Комментарий по инициаторам...")
        _ini = "\n".join([
            f"- {i['cli']} — {i['count']} обращений (инцидентов: {i['incidents']}), блоки: {i['blocks']}\n" +
            "\n".join([f"    · {s['sense']} ×{s['cnt']}" for s in i['senses'][:6]])
            for i in top_initiators_ext
        ])
        ai_comments['initiators'] = ask_analyst(
            f"Топ инициаторов обращений за период с разбивкой по сути обращений:\n\n{_ini}\n\n"
            "Прокомментируй: у кого обращения однотипные — это признак нехватки прав, инструкции или обучения, "
            "а не поломки системы; кто приносит реальные дефекты; где видно, что человек выполняет работу, "
            "которая должна быть автоматизирована. Предложи, что конкретно снимет поток от каждого из первых трёх."
        )
    else:
        ai_comments['initiators'] = "<p>Нет данных по инициаторам.</p>"

    # --- 5. Типы обращений и классификация ---
    print("🧠 [5/5] Комментарий по типам обращений...")
    _types_txt = ', '.join([f"{t['name']} — {t['count']} ({t['pct']}%)" for t in type_tiles])
    _mis_txt = "\n".join([
        f"- {m['id']} | {m['type_old']} → {m['type_new']} | {m['summary']}" for m in misclassified_rows[:30]
    ]) or 'Расхождений не найдено'
    ai_comments['types'] = ask_analyst(
        f"Распределение закрытых обращений по типам: {_types_txt}.\n"
        f"Матрица «функциональный блок × тип» содержит {len(type_matrix['rows'])} блоков.\n\n"
        f"Обращения, у которых тип проставлен неверно ({misclassified_total} шт.):\n{_mis_txt}\n\n"
        "Прокомментируй: что говорит структура типов о зрелости системы (много ли ручных правок данных "
        "против реальных инцидентов); в каких блоках тип путают чаще всего и почему; "
        "чем грозит неверная классификация для отчётности и SLA; как поправить процесс регистрации."
    )

    print(f"✅ Комментарии ИИ готовы. Потрачено токенов: {incidents_tokens['total']}")

    # Совместимость с прежним именем переменной
    incidents_analysis = ai_comments.get('inc_total', '')
else:
    incidents_analysis = ""
    ai_comments = {k: "" for k in ['inc_closed', 'inc_open', 'inc_total', 'initiators', 'types']}
    print("⏭️ ПРОПУСК: Отчет (ENABLE_MAIN = False).")

# БЛОК 5.1: ЛОГИРОВАНИЕ

In [ ]:
# =====================================================================
# [ЯЧЕЙКА 6] БЛОК 5.1: АВТОМАТИЧЕСКОЕ ЛОГИРОВАНИЕ В GOOGLE SHEETS (LOGS & TOKEN)
# =====================================================================
import datetime
import gspread
from google.colab import auth
from google.auth import default

# --- РЕЖИМ ОТЛАДКИ ---
IS_DEBUG = 0 #@param {type:"integer"}

if ENABLE_MAIN:
    if IS_DEBUG == 1:
        # 1. Авторизация в Google Sheets API
        auth.authenticate_user()
        creds, _ = default()
        gc = gspread.authorize(creds)

        # 2. Поиск и открытие файла 'Logs'
        LOGS_FILE_NAME = 'Logs'

        # ----------------------------------------------------------------------
        # ЧТО ИЗМЕНЕНО (v4.2): нет таблицы Logs — предупреждение и пропуск записи, а не падение
        # ПОЧЕМУ: прогон падал в самом конце, после всех платных вызовов ИИ
        # ----------------------------------------------------------------------
        sh = None
        try:
            sh = gc.open(LOGS_FILE_NAME)
        except Exception as e:
            print(f"⚠️ Таблица '{LOGS_FILE_NAME}' в Google Sheets не найдена или нет прав ({e}) — запись лога пропущена, отчёт строится дальше.")

    if IS_DEBUG == 1 and sh is not None:

        # 3. Генерация уникального Run ID
        run_id = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")

        # 4. Запись в лист 'Logs'
        try:
            ws_logs = sh.worksheet('Logs')
        except gspread.exceptions.WorksheetNotFound:
            ws_logs = sh.add_worksheet(title='Logs', rows=100, cols=10)
            ws_logs.append_row(['Run_ID', 'Timestamp', 'Year', 'Week', 'Total_Tickets', 'Status'])

        # ЧТО ИЗМЕНЕНО: тестовые прогоны помечаются статусом TEST
        # ПОЧЕМУ: в Logs три записи за 12.08 с одинаковым Total_Tickets=38 —
        #         тестовые запуски неотличимы от боевых
        ws_logs.append_row([
            run_id,
            datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            int(max_year),
            int(max_week),
            len(df_clean),
            "TEST" if TEST_SAMPLE_SIZE > 0 else "SUCCESS"
        ])

        # 5. Запись в лист 'Token'
        TOKEN_HEADER = [
            'Run_ID',
            'Batch_Model', 'Batch_Prompt_Tokens', 'Batch_Candidates_Tokens', 'Batch_Total_Tokens',
            'Summary_Model', 'Summary_Prompt_Tokens', 'Summary_Candidates_Tokens', 'Summary_Total_Tokens',
            'Group_Model', 'Group_Total_Tokens', 'Incidents_Total_Tokens'
        ]

        try:
            ws_tokens = sh.worksheet('Token')
        except gspread.exceptions.WorksheetNotFound:
            ws_tokens = sh.add_worksheet(title='Token', rows=100, cols=len(TOKEN_HEADER))
            ws_tokens.append_row(TOKEN_HEADER)

        # ЧТО ИЗМЕНЕНО: в шапку листа Token добавлена колонка Incidents_Total_Tokens
        # ПОЧЕМУ: появился отдельный вызов ИИ для анализа таблицы инцидентов (ячейка 5Б)
        try:
            _head = ws_tokens.row_values(1)
            if 'Incidents_Total_Tokens' not in _head:
                ws_tokens.update('A1', [TOKEN_HEADER])
                print("ℹ️ В лист 'Token' добавлены колонки Group_* / Incidents_Total_Tokens.")
        except Exception as e:
            print(f"⚠️ Не удалось обновить шапку листа 'Token': {e}")

        _group_tokens_total = group_tokens['total'] if 'group_tokens' in globals() else 0
        _incidents_tokens_total = incidents_tokens['total'] if 'incidents_tokens' in globals() else 0

        ws_tokens.append_row([
            run_id,
            BATCH_MODEL,
            total_batch_tokens['prompt'],
            total_batch_tokens['candidates'],
            total_batch_tokens['total'],
            SUMMARY_MODEL,
            summary_tokens['prompt'],
            summary_tokens['candidates'],
            summary_tokens['total'],
            SUMMARY_MODEL,
            _group_tokens_total,
            _incidents_tokens_total
        ])

        print(f"📊 Лог и токены успешно записаны в 'Logs' (Run ID: {run_id})")
    else:
        print("⏭️ ПРОПУСК: Запись в Google Sheets не выполнена (IS_DEBUG != 1 или нет таблицы Logs).")
else:
    print("⏭️ ПРОПУСК: Отчет (ENABLE_MAIN = False).")

# БЛОК 6: JINJA2 РЕНДЕРИНГ И ЭКСПОРТ

In [ ]:
# =====================================================================
# [ЯЧЕЙКА 7] БЛОК 6: JINJA2 РЕНДЕРИНГ И ЭКСПОРТ
# =====================================================================
import os
import json
import pandas as pd
from jinja2 import Template
from google.colab import drive

if ENABLE_MAIN:
    root_search_dir = '/content/drive/MyDrive/REPORTS/SPPR'

    def _pick(search_dir):
        """Ищет TEMPLATE_FILE, иначе самый свежий template*.html в каталоге."""
        if not os.path.exists(search_dir):
            return None
        wanted = str(TEMPLATE_FILE).strip().lower()
        candidates = []
        for dirpath, _, filenames in os.walk(search_dir):
            for f in filenames:
                low = f.lower()
                if low == wanted:
                    return os.path.join(dirpath, f)
                if low.startswith('template') and low.endswith('.html'):
                    candidates.append(os.path.join(dirpath, f))
        if candidates:
            newest = max(candidates, key=os.path.getmtime)
            print(f"ℹ️ {wanted} не найден, взят самый свежий шаблон: {os.path.basename(newest)}")
            return newest
        return None

    # ----------------------------------------------------------------------
    # ЧТО ИЗМЕНЕНО: шаблон сначала ищется в локальном кэше, и только потом на Диске
    # ПОЧЕМУ: если шаблон загружен вместе с выгрузкой, подключать Диск незачем
    # ----------------------------------------------------------------------
    TEMPLATE_PATH = _pick(CACHE_DIR)
    if TEMPLATE_PATH:
        print(f"🎨 Шаблон взят из кэша сессии: {os.path.basename(TEMPLATE_PATH)}")
    elif ensure_drive('нужен шаблон отчёта'):
        TEMPLATE_PATH = _pick(root_search_dir)
        if not TEMPLATE_PATH:
            print("⚠️ Шаблон не найден в кэше Диска. Сбрасываю кэш и пробую снова...")
            try:
                drive.flush_and_unmount()
            except Exception:
                pass
            drive.mount('/content/drive', force_remount=True)
            TEMPLATE_PATH = _pick(root_search_dir)
        if TEMPLATE_PATH:
            # кладём в кэш, чтобы следующий прогон обошёлся без Диска
            try:
                import shutil
                shutil.copy(TEMPLATE_PATH, os.path.join(CACHE_DIR, os.path.basename(TEMPLATE_PATH)))
            except Exception:
                pass

    if not TEMPLATE_PATH:
        raise FileNotFoundError(
            f"❌ Шаблон не найден. Ожидался {TEMPLATE_FILE} или любой template*.html "
            f"в {CACHE_DIR} либо в {root_search_dir}.\n"
            f"При DRIVE_MODE='never' выберите файл шаблона в окне загрузки вместе с выгрузкой."
        )

    print(f"✅ Шаблон найден и загружен из: {TEMPLATE_PATH}")

    with open(TEMPLATE_PATH, 'r', encoding='utf-8') as f:
        template = Environment(autoescape=True).from_string(f.read())   # v4.2: тексты обращений экранируются, HTML-блоки ИИ помечены |safe в шаблоне

    def badge_class(val):
        return 'excellent' if val == 'ОТЛИЧНО' else 'good' if val == 'ХОРОШО' else 'bad' if val == 'ПЛОХО' else 'orange'

    # Досборка недостающих переменных (используем df_clean, защищенный от nan)
    min_reg_str = df_clean['reg_dt'].min().strftime('%d.%m.%Y') if pd.notnull(df_clean['reg_dt'].min()) else 'Н/Д'
    max_reg_str = df_clean['reg_dt'].max().strftime('%d.%m.%Y') if pd.notnull(df_clean['reg_dt'].max()) else 'Н/Д'
    # ЧТО ИЗМЕНЕНО: диапазон завершения считается по done_dt, а не по cls_dt
    # ПОЧЕМУ: cls заполняется только у подтверждённых обращений, done_dt есть у всех закрытых
    max_cls_str = df_clean['done_dt'].max().strftime('%d.%m.%Y') if pd.notnull(df_clean['done_dt'].max()) else 'Н/Д'

    _dif_num_render = pd.to_numeric(df_clean['dif_Hour'], errors='coerce').dropna()
    midHour = round(float(_dif_num_render.mean()), 1) if len(_dif_num_render) else 0.0

    emo_total = len(df_clean[df_clean['tone_deviation_score'] > 0])
    emo_avg = round(df_clean[df_clean['tone_deviation_score'] > 0]['tone_deviation_score'].mean(), 1) if emo_total > 0 else 0
    emo_max = df_clean['tone_deviation_score'].max() if emo_total > 0 else 0

    alt_df = df_clean[df_clean['is_alternative']]
    alt_total = len(alt_df)
    alt_tickets = alt_df.head(10).to_dict('records')
    exc_tickets = df_clean[df_clean['final_grade'] == 'ОТЛИЧНО'].head(5).to_dict('records')
    last30_tickets = df_clean.sort_values('reg_dt', ascending=False).head(30).to_dict('records')
    hardest_tickets = df_clean.sort_values('complexity', ascending=False).head(5).to_dict('records')
    easiest_tickets = df_clean.sort_values('complexity', ascending=True).head(5).to_dict('records')

    grade_counts = pd.crosstab(df_clean['anl'], df_clean['final_grade'])
    for col in ['ОТЛИЧНО', 'ХОРОШО', 'ПЛОХО']:
        if col not in grade_counts: grade_counts[col] = 0
    analysts = grade_counts.join(df_clean.groupby('anl').agg(comp=('complexity', 'mean'), total=('id', 'count')))
    analysts['score'] = (analysts['ОТЛИЧНО'] * 3 + analysts['ХОРОШО'] * 1 - analysts['ПЛОХО'] * 3) * analysts['comp']
    analysts = analysts.reset_index().sort_values('score', ascending=False)
    top_analysts = analysts.head(10).to_dict('records')

    nom_qty = analysts.sort_values('total', ascending=False).iloc[0] if not analysts.empty else None
    nom_qual = analysts.sort_values('score', ascending=False).iloc[0] if not analysts.empty else None
    nom_hard_ticket = df_clean.sort_values('complexity', ascending=False).iloc[0] if not df_clean.empty else None

    # -----------------------------------------------------------------
    # ЧТО ИЗМЕНЕНО: бублик заменён горизонтальными столбцами (group_bars),
    #               на график динамики добавлена линия инцидентов
    # ПОЧЕМУ: круговая диаграмма нечитаема на 15 группах; инциденты нужно
    #         видеть в динамике рядом с общим потоком
    # -----------------------------------------------------------------
    history_weeks = hist_data[-6:] if 'hist_data' in globals() else []
    chart_weeks_labels = json.dumps([f"W{w['week_num']}" for w in history_weeks])
    chart_weeks_values = json.dumps([w['total_tickets'] for w in history_weeks])
    chart_weeks_incidents = json.dumps([w.get('incidents') for w in history_weeks])

    _group_tokens_total = group_tokens['total'] if 'group_tokens' in globals() else 0
    _incidents_tokens_total = incidents_tokens['total'] if 'incidents_tokens' in globals() else 0
    _groups_count = int(df_clean['topic_group'].nunique()) if 'topic_group' in df_clean.columns else 0

    # Рендеринг HTML
    html_output = template.render(
        planshet_total=planshet_total,
        planshet_closed=planshet_closed,
        week_header=WEEK_LABEL,
        week_label_short=f"Неделя {max_week}",
        week_range=f"{WEEK_START:%d.%m} — {WEEK_END:%d.%m.%Y}",
        done_total=DONE_TOTAL,
        rejected_total=REJECTED_TOTAL,
        daily_closed=daily_closed,
        daily_reg=daily_reg,
        age_bars4=age_bars4,
        inc_age_bars=inc_age_bars,
        crit_incident_rows=crit_incident_rows,
        total_tickets=total_tickets,
        med_hour=medHour,
        mid_hour=midHour,
        w1_t=w1_t,
        w1_m=w1_m,
        dates_reg=f"{min_reg_str} — {max_reg_str}",
        dates_cls=f"{min_cls_str} — {max_cls_str}",
        top_initiators=top_initiators,
        emo_total=emo_total,
        emo_avg=emo_avg,
        emo_max=emo_max,
        emo_tickets=emo_tickets,
        top_topics=top_topics,
        crit_tickets=crit_tickets,
        alt_total=alt_total,
        alt_tickets=alt_tickets,
        top_analysts=top_analysts,
        bad_tickets=bad_tickets,
        exc_tickets=exc_tickets,
        ai_conclusions=ai_conclusions,
        tip_text="Если устранить 3 самые частые проблемы, нагрузка на поддержку снизится минимум на 15%.",
        last30_tickets=last30_tickets,
        hardest_tickets=hardest_tickets,
        easiest_tickets=easiest_tickets,
        longest_tickets=longest_tickets,
        nom_hard_anl=nom_hard_ticket['anl'] if nom_hard_ticket is not None else 'Н/Д',
        nom_hard_txt=f"Тикет {nom_hard_ticket['id']} — сложность {nom_hard_ticket['complexity']}" if nom_hard_ticket is not None else 'Н/Д',
        nom_qty_anl=nom_qty['anl'] if nom_qty is not None else 'Н/Д',
        nom_qty_txt=f"{nom_qty['total']} решенных обращений" if nom_qty is not None else 'Н/Д',
        nom_qual_anl=nom_qual['anl'] if nom_qual is not None else 'Н/Д',
        nom_qual_txt=f"Итоговый балл {nom_qual['score']:.1f}" if nom_qual is not None else 'Н/Д',
        chart_weeks_labels=chart_weeks_labels,
        chart_weeks_values=chart_weeks_values,
        chart_weeks_incidents=chart_weeks_incidents,
        # -------------------------------------------------------------
        # ЧТО ИЗМЕНЕНО: переданы данные всех новых слайдов
        # ПОЧЕМУ: таймлайны, плитка типов, статистика, инициаторы, классификация,
        #         открытые обращения, инциденты, планшеты и валидация
        # -------------------------------------------------------------
        type_tiles=type_tiles,
        total_all=total_all,
        timeline_median=timeline_median,
        timeline_mean=timeline_mean,
        group_bars=group_bars,
        group_rest=group_rest,
        week_stats=week_stats,
        top_initiators_ext=top_initiators_ext,
        misclassified_rows=misclassified_rows,
        misclassified_total=misclassified_total,
        cleanup_rows=cleanup_rows,
        cleanup_total=cleanup_total,
        age_cols=age_cols,
        open_matrix_new=open_matrix_new,
        open_old_total=len(open_old),
        open_new_total=len(open_new),
        open_total=open_total,
        open_age_days=OPEN_AGE_DAYS,
        type_matrix=type_matrix,
        incidents_closed=incidents_closed,
        incidents_open=incidents_open,
        incident_rows=incident_rows,
        incident_total=incident_total,
        incident_crit=incident_crit,
        incident_open=incident_open_cnt,
        timeline_points=timeline_points,
        timeline_from=timeline_from,
        timeline_to=timeline_to,
        planshet_rows=planshet_rows,
        validation_rows=validation_rows,
        validation_ok=validation_ok,
        ai_comments=ai_comments,
        incidents_analysis=incidents_analysis,
        report_date=REPORT_DATE.strftime('%d.%m.%Y'),
        badge_class=badge_class,
        batch_tokens=total_batch_tokens['total'],
        summary_tokens=summary_tokens['total'],
        group_tokens=_group_tokens_total,
        incidents_tokens=_incidents_tokens_total,
        groups_count=_groups_count,
        batch_model=BATCH_MODEL,
        summary_model=SUMMARY_MODEL
    )

    # Базовое имя файла
    base_filename = f"Отчет по качеству W{max_week} ({min_cls_str})"
    extension = ".html"

    drive_output_path = os.path.join(TARGET_DIR, f"{base_filename}{extension}")

    # Проверка на существование файла и генерация уникального имени (_v1, _v2, ...)
    counter = 1
    while os.path.exists(drive_output_path):
        drive_output_path = os.path.join(TARGET_DIR, f"{base_filename}_v{counter}{extension}")
        counter += 1

    # Запись готового отчета
    with open(drive_output_path, 'w', encoding='utf-8') as f:
        f.write(html_output)

    print(f"✅ Отчет успешно сохранен без перезаписи: {drive_output_path}")
    files.download(drive_output_path)

    # ЧТО ИЗМЕНЕНО: история недель скачивается, если её не удалось записать на Диск
    # ПОЧЕМУ: при работе без Диска этот файл нужно подложить в следующий прогон
    _hist_cache = os.path.join(CACHE_DIR, 'history.json')
    if os.path.exists(_hist_cache) and not _DRIVE_READY['ok']:
        print("🗓️ Диск не подключался — скачиваю history.json, подложите его при следующем запуске")
        files.download(_hist_cache)
else:
    print("⏭️ ПРОПУСК: Отчет (ENABLE_MAIN = False).")

# ПРИЛОЖЕНИЕ 1 (выключено по умолчанию)

In [ ]:
# =====================================================================
# [ПРИЛОЖЕНИЕ 1] ВЕКТОРНЫЙ ПОИСК ПОХОЖИХ ОБРАЩЕНИЙ (TF-IDF)
# =====================================================================
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# 🎛️ ГЛАВНЫЙ ПЕРЕКЛЮЧАТЕЛЬ 🎛️
ENABLE_AI_ANALYSIS = False #@param {type:"boolean"}

if ENABLE_AI_ANALYSIS:
    print("🔍 Запуск алгоритма поиска похожих обращений...")

    # Заполняем пустоты
    df['desc_clean'] = df['desc'].fillna('')

    # Создаем TF-IDF матрицу
    vectorizer = TfidfVectorizer(max_features=10000, stop_words=None)
    tfidf_matrix = vectorizer.fit_transform(df['desc_clean'])

    # Вычисляем косинусное сходство
    cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)

    similar_ids_list = []
    similar_context_list = []
    SIMILARITY_THRESHOLD = 0.3 # Порог похожести

    for idx in range(len(df)):
        sim_scores = list(enumerate(cosine_sim[idx]))
        sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)[1:4] # Топ-3

        s_ids = []
        s_context = ""

        for i, score in sim_scores:
            if score >= SIMILARITY_THRESHOLD:
                match_row = df.iloc[i]
                s_ids.append(str(match_row['id']))
                s_context += f"- [ID {match_row['id']}] Проблема: {match_row['desc'][:100]}... | Решение: {match_row['sol']}\n"

        similar_ids_list.append(", ".join(s_ids) if s_ids else "Нет аналогов")
        similar_context_list.append(s_context if s_context else "Похожих инцидентов в базе не найдено.")

    # Записываем результаты в датафрейм
    df['similar_ids'] = similar_ids_list
    df['similar_past_context'] = similar_context_list

    print("✅ Поиск завершен. Контекст прошлых решений добавлен к датасету.")
else:
    print("⏭️ ПРОПУСК: Поиск похожих обращений отключен (ENABLE_AI_ANALYSIS = False).")
    # Добавляем пустые колонки, чтобы код дальше не падал, если мы передумаем
    df['similar_ids'] = "Анализ отключен"
    df['similar_past_context'] = "Анализ отключен"

In [ ]:
# =====================================================================
# [ПРИЛОЖЕНИЕ 1] БЛОК АСИНХРОННОЙ ОБРАБОТКИ ИИ (С ИСТОРИЧЕСКИМ КОНТЕКСТОМ)
# =====================================================================
class TicketResult(BaseModel):
    id: str = Field(description="Идентификатор тикета")
    topic_cluster: str = Field(description="Конкретная суть сбоя/операции. Без абстракций.")
    tone_deviation_score: int = Field(description="Шкала тона от 0 до 3")
    is_emotional: bool = Field(description="True если tone_deviation_score > 0")
    emotion_argument: str = Field(description="Обоснование оценки тона")
    complexity: int = Field(description="Сложность задачи от 1 до 5")
    initial_grade: str = Field(description="Исходная оценка")
    final_grade: str = Field(description="Итоговый грейд: ПЛОХО, ХОРОШО, ОТЛИЧНО")
    grade_argument: str = Field(description="Детальная аргументация грейда и причин снижения")
    is_alternative: bool = Field(description="True если ИИ не согласен с закрытием тикета")
    is_critical_incident: bool = Field(description="True если инцидент блокирует бизнес-процесс")
    planshet: bool = Field(description="True В теме или описании упоминается планшет (не мобильное устройство а именно планшет)")

    # ПОЛЯ ДЛЯ АНАЛИЗА РЕШЕНИЙ
    similar_ticket_ids: str = Field(description="ID похожих обращений из переданного контекста")
    is_standard_solution: bool = Field(description="True, если предложенное решение совпадает с решениями прошлых инцидентов")
    solution_comparison: str = Field(description="Сравнительный анализ: насколько текущее решение похоже на предыдущие.")

class BatchResponse(BaseModel):
    results: list[TicketResult]

SYSTEM_INSTRUCTION = """
Вы — ведущий системный аналитик и архитектор 1С. Ваша задача — провести жесткий аудит обращений.

НОВЫЕ ЗАДАЧИ:
1. Изучите блок "Похожие прошлые инциденты". Укажите их ID в similar_ticket_ids.
2. Сравните "Текущее Решение" с решениями из прошлых инцидентов.
3. Напишите емкий сравнительный анализ (solution_comparison).
4. Установите is_standard_solution = true, если подход совпадает с историческим. Если он применил костыль или контекст аналогов пуст — false.

1. ПРАВИЛА КЛАСТЕРИЗАЦИИ (topic_cluster):
- Категорически запрещены абстрактные формулировки: "Ошибка системы", "Проблема с 1С".
- Указывайте КОНКРЕТНУЮ суть сбоя.

2. КРИТЕРИИ ОЦЕНКИ ГРЕЙДА (final_grade):
- ПЛОХО: Ручные правки БД без устранения причины, формальные отписки.
- ХОРОШО: Стандартное решение, проблема решена.
- ОТЛИЧНО: Превентивное решение, инструкция, баг-репорт.

3. ШКАЛА ЭМОЦИОНАЛЬНОГО ТОНА (tone_deviation_score): от 0 до 3.
4. СЛОЖНОСТЬ (complexity): от 1 до 5.
"""

# Функции асинхронного вызова ИИ
async def call_ai_provider_batch(prompt: str):
    if AI_PROVIDER == "google":
        response = await client_gemini.aio.models.generate_content(
            model=BATCH_MODEL, contents=prompt,
            config={'system_instruction': SYSTEM_INSTRUCTION, 'response_mime_type': 'application/json', 'response_schema': BatchResponse, 'temperature': 0.1}
        )
        data = json.loads(response.text).get('results', [])
        tokens = {'total': response.usage_metadata.total_token_count if response.usage_metadata else 0}
        return data, tokens
    else:
        schema_format = BatchResponse.model_json_schema()
        full_system = f"{SYSTEM_INSTRUCTION}\nОтвет строго в JSON:\n{json.dumps(schema_format, ensure_ascii=False)}"
        response = await client_deepseek_async.chat.completions.create(
            model=BATCH_MODEL,
            messages=[{"role": "system", "content": full_system}, {"role": "user", "content": prompt}],
            response_format={"type": "json_object"}, temperature=0.1
        )
        res_raw = json.loads(response.choices[0].message.content)
        data = res_raw.get('results', []) if isinstance(res_raw, dict) else res_raw
        tokens = {'total': response.usage.total_tokens if response.usage else 0}
        return data, tokens

async def process_chunk(chunk_id, chunk, sem):
    prompt_rows = []
    for _, r in chunk.iterrows():
        prompt_rows.append(
            f"ID: {r['id']} | Инициатор: {r.get('cli','')} | Аналитик: {r.get('anl','')}\n"
            f"Тема: {r.get('sec','')}\n"
            f"Описание: {r.get('desc','')}\n"
            f"Текущее Решение: {r.get('sol','')}\n"
            f"--- ПОХОЖИЕ ПРОШЛЫЕ ИНЦИДЕНТЫ ---\n"
            f"{r.get('similar_past_context', 'Нет данных')}\n"
            f"========================================"
        )
    prompt = "Оцени пакет обращений, сделай сравнительный анализ с прошлыми решениями:\n\n" + "\n".join(prompt_rows)

    async with sem:
        try:
            await asyncio.sleep(SLEEP_TIME)
            data, tokens = await call_ai_provider_batch(prompt)
            print(f"✅ Пакет {chunk_id} завершен (Токенов: {tokens['total']})")
            return data, tokens
        except Exception as e:
            print(f"⚠️ Сбой пакета {chunk_id}: {e}")
            return [], {'total': 0}

async def run_batching():
    sem = asyncio.Semaphore(MAX_CONCURRENCY)
    tasks = []
    chunks = [df.iloc[i:i+BATCH_SIZE] for i in range(0, len(df), BATCH_SIZE)]
    print(f"🚀 Старт анализа (Пакетов: {len(chunks)})...")

    for idx, chunk in enumerate(chunks, 1):
        tasks.append(process_chunk(idx, chunk, sem))

    results = await asyncio.gather(*tasks)
    flat_results = []
    total_toks = 0
    for res_list, toks in results:
        flat_results.extend(res_list)
        total_toks += toks['total']

    print(f"✅ Анализ ИИ завершен. Итого токенов: {total_toks}")
    return flat_results, total_toks

# === ЗАПУСК ИИ ПО УСЛОВИЮ ===
if ENABLE_AI_ANALYSIS:
    ai_results, total_batch_tokens = await run_batching()
    df_final = pd.merge(df, pd.DataFrame(ai_results), on='id', how='left')
else:
    print("⏭️ ПРОПУСК: ИИ-анализ отключен (ENABLE_AI_ANALYSIS = False).")
    df_final = df.copy() # Если ИИ отключен, финализируем исходный датафрейм